<a href="https://colab.research.google.com/github/dhruvsacharya2007-del/CN_PROJECT/blob/main/InSDN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# === CELL 1  Load InSDN (manual upload / Drive, same no-API approach) ===
import pandas as pd, numpy as np, os, glob

# InSDN on Kaggle is often split into multiple CSVs (e.g. Normal + attack files)
# or a single combined CSV. Point this at your file OR a folder of InSDN CSVs.
INSDN_PATH = "/content/drive/MyDrive/CN_PROJECT/data/InSDN_Normal_and_Attack_Combined.csv"  # folder OR single .csv

def load_insdn(path):
    if os.path.isdir(path):
        csvs = sorted(glob.glob(f"{path}/*.csv"))
        assert csvs, f"No CSVs found in {path}"
        print("Found", len(csvs), "CSV(s):", [os.path.basename(c) for c in csvs])
        frames = [pd.read_csv(c, low_memory=False) for c in csvs]
        # align columns across files before concatenating
        common = set(frames[0].columns)
        for f in frames[1:]: common &= set(f.columns)
        common = list(common)
        df = pd.concat([f[common] for f in frames], ignore_index=True)
    else:
        df = pd.read_csv(path, low_memory=False)
    return df

df = load_insdn(INSDN_PATH)
df.columns = df.columns.str.strip()
print("InSDN loaded. Shape:", df.shape)

InSDN loaded. Shape: (205167, 84)


In [ ]:
# === CELL 2  Find the label column + inspect classes ===
label_col = None
for guess in ["Label", "label", "Attack", "attack", "Class", "class", "Label_Category"]:
    if guess in df.columns:
        label_col = guess; break
if label_col is None:
    label_col = df.columns[-1]     # InSDN label is typically the last column
print("Label column:", repr(label_col))
print("\nRaw class distribution:")
print(df[label_col].value_counts())

Label column: 'Label'

Raw class distribution:
Label
DDoS      73529
Normal    68424
Probe     61757
DoS        1145
BFA         295
U2R          17
Name: count, dtype: int64


In [ ]:
# === CELL 3  Harmonize InSDN labels to the SAME action-relevant classes ===
# InSDN uses different strings than CICIDS2017 (e.g. "Probe" not "Port Scanning",
# "BFA" not "Brute Force"). We map InSDN labels onto the SAME 7-way scheme your
# reward function uses, so the bandit's action space and IDEAL_ACTION stay identical
# across datasets. This is what makes the cross-dataset comparison meaningful.
#
# Adjust the left-hand strings to match EXACTLY what Cell 2 printed for your file.
INSDN_LABEL_MAP = {
    "Normal": "Normal Traffic", "BENIGN": "Normal Traffic", "Benign": "Normal Traffic",
    "Probe": "Port Scanning", "Port Scan": "Port Scanning", "Portscan": "Port Scanning",
    "DoS": "DoS",
    "DDoS": "DDoS", "DDoS (1)": "DDoS", "DDoS (2)": "DDoS", "DDoS-1": "DDoS", "DDoS-2": "DDoS",
    "BFA": "Brute Force", "Brute Force": "Brute Force", "Password": "Brute Force",
    "Web-Attack": "Web Attacks", "Web Attack": "Web Attacks", "WebAttack": "Web Attacks",
    "Botnet": "Bots", "BOTNET": "Bots", "Bot": "Bots",
    # U2R is dropped below (too few samples to be a usable class); see Cell 4.
}
df["Attack Type"] = df[label_col].map(INSDN_LABEL_MAP)

unmapped = df[df["Attack Type"].isna()][label_col].unique()
if len(unmapped):
    print("WARNING - these raw labels were NOT mapped (fix INSDN_LABEL_MAP):")
    print(" ", list(unmapped))
print("\nHarmonized class distribution:")
print(df["Attack Type"].value_counts(dropna=False))

WARNING - these raw labels were NOT mapped (fix INSDN_LABEL_MAP):
  ['U2R']

Harmonized class distribution:
Attack Type
DDoS              73529
Normal Traffic    68424
Port Scanning     61757
DoS                1145
Brute Force         295
NaN                  17
Name: count, dtype: int64


In [ ]:
# === CELL 4  Drop unmapped/tiny classes; clean numeric features ===
# U2R (~17 samples) and any unmapped rows can't support train/test stratification.
before = len(df)
df = df[df["Attack Type"].notna()].copy()
# drop classes with too few samples to stratify reliably (< 50 total)
counts = df["Attack Type"].value_counts()
tiny = counts[counts < 50].index.tolist()
if tiny:
    print("Dropping tiny classes (<50 samples):", tiny)
    df = df[~df["Attack Type"].isin(tiny)].copy()
print(f"Rows: {before} -> {len(df)} after dropping unmapped/tiny classes.")

# numeric feature matrix: drop the label columns + any non-numeric (IP, protocol str)
drop_cols = [c for c in [label_col, "Attack Type"] if c in df.columns]
X = df.drop(columns=drop_cols)
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print("Dropping non-numeric feature columns:", non_numeric[:10],
          ("..." if len(non_numeric) > 10 else ""))
    X = X.drop(columns=non_numeric)
X = X.replace([np.inf, -np.inf], np.nan)
if X.isna().any().any():
    print("Filling", int(X.isna().sum().sum()), "missing cells with column medians.")
    X = X.fillna(X.median(numeric_only=True))
print("Feature matrix:", X.shape)

Rows: 205167 -> 205150 after dropping unmapped/tiny classes.
Dropping non-numeric feature columns: ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp'] 
Feature matrix: (205150, 79)


In [ ]:
# === CELL 5  Stratified sample + encode + split (same protocol as CICIDS2017) ===
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

y_labels = df["Attack Type"].to_numpy()
SAMPLE_SIZE = min(200_000, len(X))     # same 200k dev protocol for comparability
if len(X) > SAMPLE_SIZE:
    Xs, _, ys, _ = train_test_split(X, y_labels, train_size=SAMPLE_SIZE,
                                    stratify=y_labels, random_state=42)
else:
    Xs, ys = X, y_labels

le = LabelEncoder(); y = le.fit_transform(ys)
X_train, X_test, y_train, y_test = train_test_split(
    Xs, y, test_size=0.25, stratify=y, random_state=42)
print("Classes:", list(le.classes_))
print("Train:", X_train.shape, "| Test:", X_test.shape)

Classes: ['Brute Force', 'DDoS', 'DoS', 'Normal Traffic', 'Port Scanning']
Train: (150000, 79) | Test: (50000, 79)


In [ ]:
# === CELL 6  Train the SAME lightweight RF (identical config to CICIDS2017) ===
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)
import time, joblib

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                            n_jobs=-1, random_state=42)
t0 = time.time(); rf.fit(X_train, y_train)
print(f"Trained in {time.time()-t0:.1f}s")

y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
print("="*55); print("DETECTION BASELINE - InSDN (Random Forest)"); print("="*55)
print(f"Accuracy {acc:.4f} | Precision(macro) {prec:.4f} | "
      f"Recall(macro) {rec:.4f} | F1(macro) {f1:.4f}")
print("="*55)
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

Trained in 13.0s
DETECTION BASELINE - InSDN (Random Forest)
Accuracy 0.9998 | Precision(macro) 0.9963 | Recall(macro) 0.9861 | F1(macro) 0.9910
                precision    recall  f1-score   support

   Brute Force       0.99      0.93      0.96        72
          DDoS       1.00      1.00      1.00     17921
           DoS       1.00      1.00      1.00       279
Normal Traffic       1.00      1.00      1.00     16676
 Port Scanning       1.00      1.00      1.00     15052

      accuracy                           1.00     50000
     macro avg       1.00      0.99      0.99     50000
  weighted avg       1.00      1.00      1.00     50000



In [ ]:
# === CELL 7: Save InSDN artifacts (Phase 3-5 will consume these) ===

import os
import joblib
import numpy as np

# Local artifacts directory
os.makedirs("/content/artifacts", exist_ok=True)

# Save InSDN artifacts locally
joblib.dump(rf, "/content/artifacts/rf_insdn.pkl")
joblib.dump(le, "/content/artifacts/label_encoder_insdn.pkl")
X_test.to_parquet("/content/artifacts/X_test_insdn.parquet")
np.save("/content/artifacts/y_test_insdn.npy", y_test)

# Results log
log_line = (
    f"[InSDN] Detection RF | acc {acc:.4f} | macroF1 {f1:.4f} | "
    f"classes {len(le.classes_)} | sample {len(Xs)}"
)

with open("/content/artifacts/results_log.txt", "a") as f:
    f.write(log_line + "\n")


# ============================================================
# Save to Google Drive
# My Drive/CN_PROJECT/artifacts
# ============================================================

DRIVE_DIR = "/content/drive/MyDrive/CN_PROJECT/artifacts"

try:
    os.makedirs(DRIVE_DIR, exist_ok=True)

    # Copy artifacts to Drive
    joblib.dump(rf, f"{DRIVE_DIR}/rf_insdn.pkl")
    joblib.dump(le, f"{DRIVE_DIR}/label_encoder_insdn.pkl")
    X_test.to_parquet(f"{DRIVE_DIR}/X_test_insdn.parquet")
    np.save(f"{DRIVE_DIR}/y_test_insdn.npy", y_test)

    # Append results log
    with open(f"{DRIVE_DIR}/results_log.txt", "a") as f:
        f.write(log_line + "\n")

    print("Saved InSDN artifacts to:")
    print(DRIVE_DIR)

except Exception as e:
    print("Saved locally.")
    print("Drive copy skipped:", e)


print("\nPASTE INTO RESULTS LOG:")
print(log_line)

print("""
NEXT:
Run Phase 3 (SHAP), then Phase 4 (bandit) on InSDN.

Use these artifact filenames:

rf_insdn.pkl
label_encoder_insdn.pkl
X_test_insdn.parquet
y_test_insdn.npy

The action space, IDEAL_ACTION, reward function, and bandit
code remain unchanged. This allows comparison with CICIDS2017
using the same method on a different environment.
""")

Saved InSDN artifacts to:
/content/drive/MyDrive/CN_PROJECT/artifacts

PASTE INTO RESULTS LOG:
[InSDN] Detection RF | acc 0.9998 | macroF1 0.9910 | classes 5 | sample 200000

NEXT:
Run Phase 3 (SHAP), then Phase 4 (bandit) on InSDN.

Use these artifact filenames:

rf_insdn.pkl
label_encoder_insdn.pkl
X_test_insdn.parquet
y_test_insdn.npy

The action space, IDEAL_ACTION, reward function, and bandit
code remain unchanged. This allows comparison with CICIDS2017
using the same method on a different environment.



In [ ]:
# === CELL 1  Load InSDN model + test set; build SHAP explainer ===
import importlib, subprocess, sys
for pkg in ("shap", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import shap, numpy as np, pandas as pd, os, joblib, time
os.makedirs("/content/artifacts", exist_ok=True)

CAND = ["/content/artifacts", "/content/drive/MyDrive/CN_PROJECT/artifacts"]
base = next((b for b in CAND if os.path.exists(f"{b}/rf_insdn.pkl")), None)
assert base is not None, "Run Phase 6a first - rf_insdn.pkl not found."

rf = joblib.load(f"{base}/rf_insdn.pkl")
le = joblib.load(f"{base}/label_encoder_insdn.pkl")
X_test = pd.read_parquet(f"{base}/X_test_insdn.parquet")
y_test = np.load(f"{base}/y_test_insdn.npy", allow_pickle=True)
print("InSDN classes:", list(le.classes_))
print("X_test:", X_test.shape)

explainer = shap.TreeExplainer(rf)
print("TreeExplainer built.")

KeyboardInterrupt: 

In [ ]:
# === CELL 2  SHAP attribution vectors on a stratified subset + latency ===
from sklearn.model_selection import train_test_split

EXPLAIN_N = min(3000, len(X_test))
idx = np.arange(len(X_test))
keep, _ = train_test_split(idx, train_size=EXPLAIN_N, stratify=y_test, random_state=42)
X_explain = X_test.iloc[keep].reset_index(drop=True)
y_explain = y_test[keep]

t0 = time.time()
shap_raw = np.array(explainer.shap_values(X_explain))   # (n, features, classes)
batch_secs = time.time() - t0
print("Raw SHAP shape:", shap_raw.shape,
      f"| batch {batch_secs:.1f}s ({batch_secs*1000/len(X_explain):.2f} ms/flow amortized)")

# attribution vector for the PREDICTED class (same as CICIDS2017)
pred = rf.predict(X_explain)
n, nf, nc = shap_raw.shape
attribution = np.zeros((n, nf), dtype=np.float32)
for i in range(n):
    attribution[i] = shap_raw[i, :, pred[i]]

# per-flow latency (warm-up discarded, 5 repeats, median) - same protocol
_ = explainer.shap_values(X_explain.iloc[[0]])
N_TIMED = min(200, len(X_explain)); REPEATS = 5
per_flow_ms = []
for i in range(N_TIMED):
    row = X_explain.iloc[[i]]; ts = []
    for _ in range(REPEATS):
        s = time.perf_counter(); _ = explainer.shap_values(row)
        ts.append((time.perf_counter()-s)*1000)
    per_flow_ms.append(np.median(ts))
per_flow_ms = np.array(per_flow_ms)
lat = {"mean": float(per_flow_ms.mean()), "median": float(np.median(per_flow_ms)),
       "p95": float(np.percentile(per_flow_ms, 95))}
print(f"InSDN SHAP latency: mean {lat['mean']:.2f} ms | median {lat['median']:.2f} ms | "
      f"p95 {lat['p95']:.2f} ms")

true_classes = np.array([le.classes_[c] for c in y_explain])

Raw SHAP shape: (3000, 79, 5) | batch 31.5s (10.49 ms/flow amortized)
InSDN SHAP latency: mean 10.69 ms | median 10.22 ms | p95 14.44 ms


In [ ]:
# === CELL 3  Action space + reward (IDENTICAL to CICIDS2017) ===
# Note: InSDN here has 5 classes (no Web Attacks / Bots in this release). The
# IDEAL_ACTION dict still covers all 7; only the present classes are used.
ACTIONS = ["allow", "rate_limit", "isolate", "block"]
IDEAL_ACTION = {"Normal Traffic":"allow","Port Scanning":"rate_limit","Brute Force":"isolate",
                "Web Attacks":"isolate","Bots":"isolate","DoS":"block","DDoS":"block"}
SEVERITY = {"allow":0,"rate_limit":1,"isolate":2,"block":3}
def compute_reward(tc, a):
    ideal = IDEAL_ACTION[tc]
    if a == ideal: return 1.0
    if tc == "Normal Traffic" and a != "allow": return -1.0
    if tc != "Normal Traffic" and a == "allow": return -1.0
    return max(-0.5, 0.5 - 0.5*abs(SEVERITY[a]-SEVERITY[ideal]))
print("Reward + action space ready (identical to CICIDS2017).")
print("Classes present in InSDN attributions:", sorted(set(true_classes)))

Reward + action space ready (identical to CICIDS2017).
Classes present in InSDN attributions: [np.str_('Brute Force'), np.str_('DDoS'), np.str_('DoS'), np.str_('Normal Traffic'), np.str_('Port Scanning')]


In [ ]:
# === CELL 4  LinUCB (identical, alpha=1.0) ===
class LinUCB:
    def __init__(self, n_actions, d, alpha=1.0):
        self.n_actions=n_actions; self.A=[np.identity(d) for _ in range(n_actions)]
        self.b=[np.zeros((d,1)) for _ in range(n_actions)]; self.alpha=alpha
    def select(self, x):
        x=x.reshape(-1,1); ba,bp=0,-1e18
        for a in range(self.n_actions):
            Ai=np.linalg.inv(self.A[a]); th=Ai@self.b[a]
            p=float((th.T@x).item())+self.alpha*float(np.sqrt((x.T@Ai@x).item()))
            if p>bp: bp,ba=p,a
        return ba
    def update(self,a,x,r): x=x.reshape(-1,1); self.A[a]+=x@x.T; self.b[a]+=r*x
    def act_greedy(self,x):
        x=x.reshape(-1,1); ba,bp=0,-1e18
        for a in range(self.n_actions):
            th=np.linalg.inv(self.A[a])@self.b[a]; p=float((th.T@x).item())
            if p>bp: bp,ba=p,a
        return ba

In [ ]:
# === CELL 5  100-seed bandit vs honest static (identical protocol) ===
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
from scipy import stats

dominant_feat = np.abs(attribution).argmax(axis=1)

def run_once(seed):
    tr, te = train_test_split(np.arange(n), test_size=0.30,
                              stratify=true_classes, random_state=seed)
    Xc = StandardScaler().fit_transform(attribution).astype(np.float64)

    # honest static baseline
    bar = defaultdict(lambda: np.zeros(len(ACTIONS)))
    bac = defaultdict(lambda: np.zeros(len(ACTIONS)))
    for i in tr:
        f = dominant_feat[i]
        for ai,a in enumerate(ACTIONS):
            bar[f][ai] += compute_reward(true_classes[i], a); bac[f][ai] += 1
    smap = {f:int(np.argmax(bar[f]/np.maximum(bac[f],1))) for f in bar}
    def sact(i): return ACTIONS[smap.get(dominant_feat[i], 1)]

    # bandit, single pass
    bandit = LinUCB(len(ACTIONS), Xc.shape[1], alpha=1.0)
    rng = np.random.default_rng(seed)
    for i in rng.permutation(tr):
        a = bandit.select(Xc[i]); bandit.update(a, Xc[i], compute_reward(true_classes[i], ACTIONS[a]))

    def ev(fn):
        rs, cor, fp, nn = [], 0, 0, 0
        for i in te:
            a = fn(i); rs.append(compute_reward(true_classes[i], a))
            cor += (a == IDEAL_ACTION[true_classes[i]])
            if true_classes[i]=="Normal Traffic":
                nn += 1; fp += (a!="allow")
        return cor/len(te), float(np.mean(rs)), fp/max(1,nn)
    s = ev(lambda i: sact(i))
    b = ev(lambda i: ACTIONS[bandit.act_greedy(Xc[i])])
    return dict(s_acc=s[0], s_rew=s[1], s_fp=s[2], b_acc=b[0], b_rew=b[1], b_fp=b[2])

res = pd.DataFrame([{**run_once(s), "seed":s} for s in range(100)])
print("100 seeds complete.")

100 seeds complete.


In [ ]:
# === CELL 6  Aggregate + significance (identical reporting) ===
def ci95(x):
    x=np.asarray(x); se=stats.sem(x); h=se*stats.t.ppf(0.975, len(x)-1)
    return (x.mean()-h, x.mean()+h)
def fmt(c):
    lo,hi=ci95(res[c]); return f"{res[c].mean():.4f} ± {res[c].std():.4f} (95% CI [{lo:.4f},{hi:.4f}])"

wins = int((res["b_rew"]>res["s_rew"]).sum())
w_rew = stats.wilcoxon(res["b_rew"], res["s_rew"])
w_acc = stats.wilcoxon(res["b_acc"], res["s_acc"])

print("="*72)
print("InSDN - MITIGATION: bandit vs honest static (100 seeds)")
print("="*72)
for m,label in [("acc","ACCURACY"),("rew","REWARD"),("fp","FP_RATE")]:
    print(f"\n{label}")
    print(f"  static: {fmt('s_'+m)}")
    print(f"  bandit: {fmt('b_'+m)}")
print("\n"+"="*72)
print(f"Bandit beats static on reward: {wins}/100 seeds")
print(f"Wilcoxon p(reward)={w_rew.pvalue:.2e} | p(accuracy)={w_acc.pvalue:.2e}")
print("="*72)

InSDN - MITIGATION: bandit vs honest static (100 seeds)

ACCURACY
  static: 0.7827 ± 0.0080 (95% CI [0.7811,0.7843])
  bandit: 0.9952 ± 0.0020 (95% CI [0.9948,0.9956])

REWARD
  static: 0.6622 ± 0.0130 (95% CI [0.6597,0.6648])
  bandit: 0.9927 ± 0.0036 (95% CI [0.9920,0.9935])

FP_RATE
  static: 0.0387 ± 0.0093 (95% CI [0.0368,0.0405])
  bandit: 0.0019 ± 0.0039 (95% CI [0.0012,0.0027])

Bandit beats static on reward: 100/100 seeds
Wilcoxon p(reward)=3.88e-18 | p(accuracy)=3.85e-18


In [ ]:
# === CELL 7  Save + results-log lines (InSDN) — self-contained ===
import os
from scipy import stats

# recompute what the earlier cell would have defined
wins  = int((res["b_rew"] > res["s_rew"]).sum())
w_rew = stats.wilcoxon(res["b_rew"], res["s_rew"])

os.makedirs("/content/artifacts", exist_ok=True)
res.to_csv("/content/artifacts/phase6b_insdn_multiseed.csv", index=False)

lines = []
if "lat" in globals() and "X_explain" in globals():
    lines.append(f"[InSDN] SHAP latency | mean {lat['mean']:.2f} ms "
                 f"median {lat['median']:.2f} ms p95 {lat['p95']:.2f} ms | "
                 f"explained {len(X_explain)} flows")
else:
    print("(skipping the latency line - 'lat' not in this session)")

lines.append(f"[InSDN] Phase mitigation {len(res)}-seed | "
             f"STATIC acc {res['s_acc'].mean():.4f}±{res['s_acc'].std():.4f} "
             f"rew {res['s_rew'].mean():.4f}±{res['s_rew'].std():.4f} "
             f"FP {res['s_fp'].mean():.4f} || "
             f"BANDIT acc {res['b_acc'].mean():.4f}±{res['b_acc'].std():.4f} "
             f"rew {res['b_rew'].mean():.4f}±{res['b_rew'].std():.4f} "
             f"FP {res['b_fp'].mean():.4f} | "
             f"gap {(res['b_rew'].mean()-res['s_rew'].mean())*100:+.2f}pp | "
             f"wins {wins}/{len(res)} | Wilcoxon p(rew)={w_rew.pvalue:.2e}")

for d in ("/content/artifacts", "/content/drive/MyDrive/CN_PROJECT/artifacts"):
    try:
        os.makedirs(d, exist_ok=True)
        res.to_csv(f"{d}/phase6b_insdn_multiseed.csv", index=False)
        with open(f"{d}/results_log.txt", "a") as f:
            f.write("\n".join(lines) + "\n")
    except Exception as e:
        print(f"(skipped {d}: {e})")

print("\nPASTE INTO RESULTS LOG:")
for l in lines: print(" ", l)


PASTE INTO RESULTS LOG:
  [InSDN] SHAP latency | mean 10.69 ms median 10.22 ms p95 14.44 ms | explained 3000 flows
  [InSDN] Phase mitigation 100-seed | STATIC acc 0.7827±0.0080 rew 0.6622±0.0130 FP 0.0387 || BANDIT acc 0.9952±0.0020 rew 0.9927±0.0036 FP 0.0019 | gap +33.05pp | wins 100/100 | Wilcoxon p(rew)=3.88e-18


In [ ]:
# ============================================================
# DIAGNOSTIC - Why does the static baseline collapse on InSDN?
# ============================================================
# The static baseline keys on each flow's SINGLE dominant SHAP feature
# (argmax|attribution|). If InSDN's attributions concentrate on very few
# dominant features, the fixed lookup can't separate the classes -> it tanks.
# This cell CHECKS that, so you can EXPLAIN the 21-point gap in the paper
# instead of just asserting it. Run after Phase 6b (uses `attribution`,
# `true_classes`, `X_explain`, `ACTIONS`, `IDEAL_ACTION`, `compute_reward`).
# ============================================================

import numpy as np, pandas as pd
from collections import defaultdict, Counter

dominant_feat = np.abs(attribution).argmax(axis=1)
feat_names = list(X_explain.columns)
n_features = attribution.shape[1]

# 1) How concentrated are the dominant features?
counts = Counter(dominant_feat)
n_distinct = len(counts)
top5 = counts.most_common(5)
print("="*64)
print("DOMINANT-FEATURE CONCENTRATION (drives the static baseline)")
print("="*64)
print(f"Features available:            {n_features}")
print(f"Distinct dominant features:    {n_distinct}")
print(f"Top-5 dominant features (feature_idx -> #flows where it dominates):")
for fi, c in top5:
    print(f"   {feat_names[fi][:40]:42s} {c:5d} flows ({100*c/len(dominant_feat):.1f}%)")
share_top1 = top5[0][1]/len(dominant_feat)
print(f"\nSingle most-common dominant feature covers {share_top1*100:.1f}% of ALL flows.")

# 2) Is the dominant feature ambiguous across classes?
# For each dominant feature, how many DIFFERENT ideal-actions do the flows want?
# If one feature dominates flows that need DIFFERENT actions, a fixed lookup on
# that feature MUST be wrong for some of them - that's the mechanism.
print("\n" + "="*64)
print("ACTION AMBIGUITY per dominant feature")
print("="*64)
feat_actions = defaultdict(Counter)
for i in range(len(dominant_feat)):
    ideal = IDEAL_ACTION[true_classes[i]]
    feat_actions[dominant_feat[i]][ideal] += 1

ambiguous = 0
covered = 0
for fi, actions in sorted(feat_actions.items(), key=lambda kv: -sum(kv[1].values())):
    total = sum(actions.values())
    covered += total
    if len(actions) > 1:
        ambiguous += total
    if covered <= 0.9*len(dominant_feat) and len(actions) > 1:
        # show the big ambiguous buckets
        share = 100*total/len(dominant_feat)
        if share > 2:
            dist = ", ".join(f"{a}:{c}" for a,c in actions.most_common())
            print(f"  {feat_names[fi][:34]:36s} {total:5d} flows -> {dist}")
print(f"\nFlows whose dominant feature maps to MORE THAN ONE ideal action: "
      f"{ambiguous}/{len(dominant_feat)} = {100*ambiguous/len(dominant_feat):.1f}%")
print("(These are flows the single-feature static rule CANNOT get right -> "
      "this is why it caps around its observed accuracy.)")

# 3) The honest takeaway line for the paper
print("\n" + "="*64)
print("PAPER-READY EXPLANATION")
print("="*64)
print(f"""On InSDN, {100*ambiguous/len(dominant_feat):.0f}% of flows have a dominant SHAP
feature that is shared across classes requiring different mitigation actions.
A single-feature static rule is therefore structurally unable to separate them,
capping its accuracy. The bandit uses the FULL attribution vector and is not
limited this way - explaining the large ({{gap}}-point) accuracy gap on InSDN
versus the smaller gap on CICIDS2017, where dominant features are more
class-discriminative.""")
print("Fill {gap} with the actual number from your results (~21).")


DOMINANT-FEATURE CONCENTRATION (drives the static baseline)
Features available:            79
Distinct dominant features:    9
Top-5 dominant features (feature_idx -> #flows where it dominates):
   Src Port                                    1652 flows (55.1%)
   Pkt Len Max                                  863 flows (28.8%)
   Dst Port                                     333 flows (11.1%)
   Init Bwd Win Byts                            127 flows (4.2%)
   Bwd Header Len                                14 flows (0.5%)

Single most-common dominant feature covers 55.1% of ALL flows.

ACTION AMBIGUITY per dominant feature
  Src Port                              1652 flows -> block:1077, rate_limit:572, isolate:3
  Pkt Len Max                            863 flows -> allow:854, block:9

Flows whose dominant feature maps to MORE THAN ONE ideal action: 2989/3000 = 99.6%
(These are flows the single-feature static rule CANNOT get right -> this is why it caps around its observed accuracy.)

PAPER

In [ ]:
import glob
for b in ["/content/artifacts", "/content/drive/MyDrive/CN_PROJECT/artifacts"]:
    print(b, sorted(glob.glob(f"{b}/shap_attributions*.parquet")))

/content/artifacts []
/content/drive/MyDrive/CN_PROJECT/artifacts ['/content/drive/MyDrive/CN_PROJECT/artifacts/shap_attributions_cicids2017.parquet', '/content/drive/MyDrive/CN_PROJECT/artifacts/shap_attributions_xgb_cicids2017.parquet', '/content/drive/MyDrive/CN_PROJECT/artifacts/shap_attributions_xgb_ciciot2023.parquet', '/content/drive/MyDrive/CN_PROJECT/artifacts/shap_attributions_xgb_insdn.parquet']


In [ ]:
# ============================================================
# EBM-DRIVEN BANDIT  -  run the closed loop with NO SHAP in it
#
# Today: XGBoost -> SHAP -> attribution vector -> LinUCB -> OpenFlow rule.
# This cell replaces the middle step:
#          EBM -> its own additive terms -> LinUCB -> OpenFlow rule.
#
# Both are run here on the SAME flows and the SAME 100 seeds, so the
# comparison is paired and head-to-head.
#
# FOR InSDN ONLY. Paste as ONE cell in the InSDN notebook,
# after its Phase 1-3 cells have run in this session.
# Needs in memory: X_train, X_test, y_train, y_test, le
# Needs on disk:   shap_attributions*.parquet  (from Phase 3)
#
# Runtime: ~15-25 min (EBM training + 2 x 100 seeds).
# ============================================================

DATASET  = "InSDN"
SEEDS    = 100
EBM_K    = 0        # pairwise terms; 30 was the knee on CICIDS2017, 0 on InSDN
FIT_N    = 60000     # EBM training rows (match the glass-box comparison)
# ============================================================

import importlib, subprocess, sys
for _p, _pip in (("interpret", "interpret-core"),):
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pip], check=True)

import numpy as np, pandas as pd, os, glob, time
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score
from scipy import stats
from interpret.glassbox import ExplainableBoostingClassifier

for _v in ["X_train", "X_test", "y_train", "y_test", "le"]:
    assert _v in globals(), f"{_v} missing - re-run the Phase 2 cells first."

CAND = ["/content/artifacts", "/content/drive/MyDrive/CN_PROJECT/artifacts"]
TAG  = DATASET.lower().replace(" ", "")
N_FEATURES, N_CLASSES = X_train.shape[1], len(le.classes_)

# ---------------- the SHAP attributions from Phase 3 ----------------
attr_path = "/content/drive/MyDrive/CN_PROJECT/artifacts/shap_attributions_xgb_insdn.parquet"
assert os.path.exists(attr_path), (
    f"Not found: {attr_path}\n"
    "Run the InSDN Phase 3 cells first, or check the filename.")
attr_df = pd.read_parquet(attr_path)
shap_cols = [c for c in attr_df.columns if c.startswith("shap_")]
A_shap    = attr_df[shap_cols].to_numpy()
true_classes = attr_df["true_class"].to_numpy()
EXPLAIN_N = len(attr_df)
print(f"SHAP attributions: {os.path.basename(attr_path)}  "
      f"({EXPLAIN_N:,} flows x {len(shap_cols)} features)")

# ---------------- recover the SAME flows Phase 3 explained ----------------
# Phase 3 used: train_test_split(arange(len(X_test)), train_size=EXPLAIN_N,
#                                stratify=y_test, random_state=42)
y_test_arr = np.asarray(y_test)
try:
    keep_idx, _ = train_test_split(np.arange(len(X_test)), train_size=EXPLAIN_N,
                                   stratify=y_test_arr, random_state=42)
except ValueError:
    keep_idx, _ = train_test_split(np.arange(len(X_test)), train_size=EXPLAIN_N,
                                   random_state=42)
X_explain = X_test.iloc[keep_idx]
recovered = np.array([le.classes_[c] for c in y_test_arr[keep_idx]])
match = float((recovered == true_classes).mean())
print(f"recovered the same explained rows: {match*100:.1f}% class agreement")
assert match > 0.99, (
    "Could not reproduce Phase 3's explained subset - the comparison would not "
    "be paired. Check that EXPLAIN_N and random_state=42 match Phase 3.")

# ---------------- train the EBM on the same rows as the comparison ----------------
if FIT_N and FIT_N < len(X_train):
    idx = np.random.default_rng(42).choice(len(X_train), FIT_N, replace=False)
    Xtr, ytr = X_train.iloc[idx].reset_index(drop=True), np.asarray(y_train)[idx]
else:
    Xtr, ytr = X_train.reset_index(drop=True), np.asarray(y_train)
print(f"\ntraining EBM (K={EBM_K}) on {len(Xtr):,} rows ...")
sw  = compute_sample_weight(class_weight="balanced", y=ytr)
t0  = time.time()
ebm = ExplainableBoostingClassifier(interactions=EBM_K, max_rounds=200,
                                    outer_bags=4, n_jobs=-1, random_state=42)
ebm.fit(Xtr, ytr, sample_weight=sw)
print(f"   fit {time.time()-t0:.0f}s | acc "
      f"{accuracy_score(y_test_arr, ebm.predict(X_test)):.4f} | macro-F1 "
      f"{f1_score(y_test_arr, ebm.predict(X_test), average='macro', zero_division=0):.4f}")

# ---------------- EBM attribution vector, same shape as the SHAP one ----------------
def terms_to_features(model, ts):
    """Collapse (n, terms, classes) onto (n, FEATURES, classes).
    A pair term's contribution is split evenly between its two features -
    a modelling choice, stated in the paper."""
    ts = np.asarray(ts)
    if ts.ndim == 2: ts = ts[:, :, None]
    out = np.zeros((ts.shape[0], N_FEATURES, ts.shape[2]))
    for t, feats in enumerate(model.term_features_):
        share = 1.0 / len(feats)
        for fi in feats:
            out[:, fi, :] += ts[:, t, :] * share
    return out

print("computing EBM attributions ...")
T = terms_to_features(ebm, ebm.eval_terms(X_explain))      # (n, features, classes)
pred_ebm = ebm.predict(X_explain)
A_ebm = np.stack([T[i, :, pred_ebm[i]] for i in range(len(X_explain))])
assert A_ebm.shape == A_shap.shape, (A_ebm.shape, A_shap.shape)
print(f"   EBM attribution matrix {A_ebm.shape}  (same shape as SHAP's)")

# ---------------- the shared protocol ----------------
ACTIONS  = ["allow", "rate_limit", "isolate", "block"]
SEVERITY = {"allow": 0, "rate_limit": 1, "isolate": 2, "block": 3}
IDEAL = {"Normal Traffic": "allow", "Port Scanning": "rate_limit",
         "Brute Force": "isolate", "Web Attacks": "isolate", "Bots": "isolate",
         "DoS": "block", "DDoS": "block", "MITM": "isolate",
         "DNS Spoofing": "isolate"}
for c in np.unique(true_classes): IDEAL.setdefault(c, "isolate")

def reward(tc, a):
    ideal = IDEAL[tc]
    if a == ideal: return 1.0
    if tc == "Normal Traffic" and a != "allow": return -1.0
    if tc != "Normal Traffic" and a == "allow": return -1.0
    return max(-0.5, 0.5 - 0.5*abs(SEVERITY[a] - SEVERITY[ideal]))

class LinUCB:
    def __init__(self, n, d, alpha=1.0):
        self.n, self.alpha = n, alpha
        self.A = [np.identity(d) for _ in range(n)]
        self.b = [np.zeros((d, 1)) for _ in range(n)]
    def select(self, x):
        x = x.reshape(-1, 1); best, bp = 0, -1e18
        for a in range(self.n):
            Ai = np.linalg.inv(self.A[a]); th = Ai @ self.b[a]
            p = float((th.T@x).item()) + self.alpha*float(np.sqrt((x.T@Ai@x).item()))
            if p > bp: bp, best = p, a
        return best
    def update(self, a, x, r):
        x = x.reshape(-1, 1); self.A[a] += x@x.T; self.b[a] += r*x
    def greedy(self, x):
        x = x.reshape(-1, 1); best, bp = 0, -1e18
        for a in range(self.n):
            p = float(((np.linalg.inv(self.A[a])@self.b[a]).T@x).item())
            if p > bp: bp, best = p, a
        return best

def run_once(seed, A):
    """One complete experiment on attribution matrix A."""
    ctx = StandardScaler().fit_transform(A).astype(np.float64)
    dom = np.abs(A).argmax(axis=1)
    idx = np.arange(len(A))
    try:
        tr, te = train_test_split(idx, test_size=0.30, stratify=true_classes,
                                  random_state=seed)
    except ValueError:
        tr, te = train_test_split(idx, test_size=0.30, random_state=seed)

    bar = defaultdict(lambda: np.zeros(len(ACTIONS)))
    bac = defaultdict(lambda: np.zeros(len(ACTIONS)))
    for i in tr:
        f = dom[i]
        for ai, a in enumerate(ACTIONS):
            bar[f][ai] += reward(true_classes[i], a); bac[f][ai] += 1
    smap = {f: int(np.argmax(bar[f]/np.maximum(bac[f], 1))) for f in bar}

    bandit = LinUCB(len(ACTIONS), ctx.shape[1])
    for i in np.random.default_rng(seed).permutation(tr):
        a = bandit.select(ctx[i])
        bandit.update(a, ctx[i], reward(true_classes[i], ACTIONS[a]))

    def ev(pick):
        rs, cor, fp, nn = [], 0, 0, 0
        for i in te:
            a = pick(i)
            rs.append(reward(true_classes[i], a))
            cor += (a == IDEAL[true_classes[i]])
            if true_classes[i] == "Normal Traffic":
                nn += 1; fp += (a != "allow")
        return cor/len(te), float(np.mean(rs)), fp/max(1, nn)

    s = ev(lambda i: ACTIONS[smap.get(dom[i], 1)])
    b = ev(lambda i: ACTIONS[bandit.greedy(ctx[i])])
    return dict(s_acc=s[0], s_rew=s[1], s_fp=s[2],
                b_acc=b[0], b_rew=b[1], b_fp=b[2], buckets=len(smap))

# ---------------- run both ----------------
out = {}
for name, A in (("SHAP-driven (current)", A_shap), ("EBM-driven (no SHAP)", A_ebm)):
    print(f"\n{name}: {SEEDS} seeds ...")
    out[name] = pd.DataFrame([run_once(s, A) for s in range(SEEDS)])
    r = out[name]
    print(f"   bandit rew {r.b_rew.mean():.4f} | static rew {r.s_rew.mean():.4f} "
          f"| bandit FP {r.b_fp.mean():.4f} | static buckets {r.buckets.mean():.1f}")

rows = []
for name, r in out.items():
    rows.append({"Attribution source": name,
                 "bandit reward": round(r.b_rew.mean(), 4),
                 "static reward": round(r.s_rew.mean(), 4),
                 "gap pp": round((r.b_rew.mean()-r.s_rew.mean())*100, 2),
                 "bandit acc": round(r.b_acc.mean(), 4),
                 "bandit FP": round(r.b_fp.mean(), 4),
                 "static FP": round(r.s_fp.mean(), 4),
                 "static buckets": round(r.buckets.mean(), 1),
                 "wins": f"{int((r.b_rew>r.s_rew).sum())}/{SEEDS}",
                 "p(reward)": f"{stats.wilcoxon(r.b_rew, r.s_rew).pvalue:.2e}"})
t = pd.DataFrame(rows)

print("\n" + "=" * 126)
print(f"CLOSED LOOP WITH AND WITHOUT SHAP - {DATASET}, {SEEDS} paired seeds")
print("=" * 126)
print(t.to_string(index=False))
print("=" * 126)

S, E = out["SHAP-driven (current)"], out["EBM-driven (no SHAP)"]
p_pair = stats.wilcoxon(E.b_rew, S.b_rew).pvalue
d = (E.b_rew.mean() - S.b_rew.mean()) * 100
print(f"\nBANDIT vs BANDIT (paired over the same seeds)")
print(f"  EBM-driven minus SHAP-driven: {d:+.2f} pp reward   (Wilcoxon p = {p_pair:.2e})")
print(f"  benign FP: {S.b_fp.mean():.4f} (SHAP) -> {E.b_fp.mean():.4f} (EBM)")

print("\nVERDICT")
if abs(d) < 0.5:
    print(f"  The loop performs the same without SHAP ({d:+.2f} pp). You can state")
    print("  that the glass-box attribution is a drop-in replacement for SHAP in")
    print("  the decision layer, at roughly a quarter of the explanation cost.")
    print("  THIS is the result that makes the glass-box your pipeline, not an aside.")
elif d < 0:
    print(f"  The loop is WORSE without SHAP ({d:+.2f} pp). Report it: the glass-box")
    print("  is cheaper to explain but its attributions carry less decision-relevant")
    print("  information. The cost saving is real; it is not free.")
else:
    print(f"  The loop is BETTER without SHAP ({d:+.2f} pp). Check the bucket counts")
    print("  below before claiming this - a weaker static baseline can flatter the")
    print("  bandit rather than the attributions being better.")

sb_s, sb_e = S.buckets.mean(), E.buckets.mean()
print(f"\nFAIRNESS CHECK - distinct dominant-feature buckets in the static baseline")
print(f"  SHAP {sb_s:.1f}   EBM {sb_e:.1f}")
if sb_e < sb_s * 0.6:
    print("  !! The EBM static baseline has far fewer buckets, so it is a WEAKER")
    print("     opponent. Any bandit 'win' on EBM attributions is partly an artifact")
    print("     of that. Compare bandit-to-bandit (the paired figure above), not")
    print("     bandit-to-its-own-static, and say so explicitly in the paper.")
else:
    print("  Bucket counts are comparable, so the two static baselines are a fair")
    print("  match and either comparison is sound.")

for d_ in CAND:
    try:
        os.makedirs(d_, exist_ok=True)
        t.to_csv(f"{d_}/ebm_bandit_{TAG}.csv", index=False)
        S.to_csv(f"{d_}/ebm_bandit_seeds_shap_{TAG}.csv", index=False)
        E.to_csv(f"{d_}/ebm_bandit_seeds_ebm_{TAG}.csv", index=False)
    except Exception: pass
log = (f"[EBM-BANDIT/{DATASET}] K={EBM_K} | SHAP-driven rew {S.b_rew.mean():.4f} "
       f"FP {S.b_fp.mean():.4f} | EBM-driven rew {E.b_rew.mean():.4f} "
       f"FP {E.b_fp.mean():.4f} | delta {d:+.2f}pp p={p_pair:.2e} | "
       f"buckets {sb_s:.1f} vs {sb_e:.1f} | n={EXPLAIN_N} seeds={SEEDS}")
for d_ in CAND:
    try:
        with open(f"{d_}/results_log.txt", "a") as fh: fh.write(log + "\n")
    except Exception: pass
print("\nPASTE INTO RESULTS LOG:\n ", log)

SHAP attributions: shap_attributions_xgb_insdn.parquet  (3,000 flows x 79 features)
recovered the same explained rows: 100.0% class agreement

training EBM (K=0) on 60,000 rows ...
   fit 141s | acc 0.9999 | macro-F1 0.9955
computing EBM attributions ...
   EBM attribution matrix (3000, 79)  (same shape as SHAP's)

SHAP-driven (current): 100 seeds ...
   bandit rew 0.9955 | static rew 0.9921 | bandit FP 0.0000 | static buckets 10.0

EBM-driven (no SHAP): 100 seeds ...
   bandit rew 0.9950 | static rew 0.9921 | bandit FP 0.0012 | static buckets 7.0

CLOSED LOOP WITH AND WITHOUT SHAP - InSDN, 100 paired seeds
   Attribution source  bandit reward  static reward  gap pp  bandit acc  bandit FP  static FP  static buckets   wins p(reward)
SHAP-driven (current)         0.9955         0.9921    0.35      0.9970     0.0000        0.0            10.0 88/100  6.10e-14
 EBM-driven (no SHAP)         0.9950         0.9921    0.30      0.9967     0.0012        0.0             7.0 76/100  8.50e-12

BAN

In [ ]:
# ============================================================
# FULL MODEL SWAP -> XGBoost   [InSDN]
# Paste as ONE cell at the end of the InSDN notebook and run.
#
# Needs in memory: X_train, X_test, y_train, y_test, le
# (i.e. Phase 1 + Phase 2 cells already run)
#
# Re-runs the WHOLE downstream chain on XGBoost:
#   detection -> SHAP attributions -> latency -> 100-seed bandit vs static
# so every number is internally consistent with the new detector.
#
# Runtime: roughly 5-8 min (XGB train ~82s, SHAP ~20s, 100 seeds a few min).
# ============================================================

import importlib, subprocess, sys
for _pkg in ("xgboost", "shap", "scipy"):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

import numpy as np, pandas as pd, time, os, joblib, shap
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from collections import defaultdict
from scipy import stats

for _v in ["X_train", "X_test", "y_train", "y_test", "le"]:
    assert _v in globals(), f"{_v} missing - re-run Phase 1 + Phase 2 first."

os.makedirs("/content/artifacts", exist_ok=True)
DRIVE_DIR = "/content/drive/MyDrive/CN_PROJECT/artifacts"
n_classes = len(le.classes_)
print(f"XGBoost swap on {len(X_train):,} train / {len(X_test):,} test, "
      f"{X_train.shape[1]} features, {n_classes} classes.\n")


# ============================================================
# 1. DETECTION - XGBoost replaces Random Forest
# ============================================================
# RF used class_weight="balanced"; XGBoost's multiclass equivalent is
# per-sample weights, so we compute the identical balanced weighting.
sw = compute_sample_weight(class_weight="balanced", y=y_train)

xgb = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.3,
    tree_method="hist", n_jobs=-1,
    objective="multi:softprob", num_class=n_classes,
    random_state=42, verbosity=0,
)
t0 = time.time(); xgb.fit(X_train, y_train, sample_weight=sw)
train_s = time.time() - t0

y_pred = xgb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average="macro", zero_division=0)
print("="*60)
print("DETECTION - InSDN (XGBoost)")
print("="*60)
print(f"Accuracy {acc:.4f} | Macro-F1 {f1:.4f} | trained in {train_s:.1f}s")
print("="*60)
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))


# ============================================================
# 2. SHAP ATTRIBUTIONS + LATENCY
# ============================================================
explainer = shap.TreeExplainer(xgb)

def safe_split(idx, strat, **kw):
    """Stratify when possible; fall back to plain split if a class is too rare.
    Rare classes (e.g. Bots ~193 of 250k) can round to <2 members in a small
    stratified sample, which makes sklearn raise."""
    try:
        return train_test_split(idx, stratify=strat, **kw)
    except ValueError as e:
        print(f"  [stratify fallback] {e}")
        return train_test_split(idx, **kw)

EXPLAIN_N = min(3000, len(X_test))
keep, _ = safe_split(np.arange(len(X_test)), y_test,
                     train_size=EXPLAIN_N, random_state=42)
X_explain = X_test.iloc[keep].reset_index(drop=True)
y_explain = y_test[keep]

t0 = time.time()
shap_raw = np.array(explainer.shap_values(X_explain))   # (n, features, classes)
batch_s = time.time() - t0
print(f"\nSHAP batch over {EXPLAIN_N} flows: {batch_s:.1f}s "
      f"({batch_s*1000/EXPLAIN_N:.2f} ms/flow amortized) | shape {shap_raw.shape}")

# attribution vector = SHAP values for the PREDICTED class (same rule as before)
pred = xgb.predict(X_explain)
attribution = np.zeros((shap_raw.shape[0], shap_raw.shape[1]), dtype=np.float32)
for i in range(shap_raw.shape[0]):
    attribution[i] = shap_raw[i, :, pred[i]]
true_classes = np.array([le.classes_[c] for c in y_explain])

# per-flow latency: warm-up discarded, repeats, median
_ = explainer.shap_values(X_explain.iloc[[0]])
N_TIMED, REPEATS = 200, 5
per_flow = []
for i in range(min(N_TIMED, len(X_explain))):
    row = X_explain.iloc[[i]]; reps = []
    for _ in range(REPEATS):
        t = time.perf_counter(); _ = explainer.shap_values(row)
        reps.append((time.perf_counter()-t)*1000)
    per_flow.append(np.median(reps))
per_flow = np.array(per_flow)
lat = {"mean": float(per_flow.mean()), "median": float(np.median(per_flow)),
       "p95": float(np.percentile(per_flow, 95))}
print(f"SHAP per-flow latency: mean {lat['mean']:.2f} ms | "
      f"median {lat['median']:.2f} ms | p95 {lat['p95']:.2f} ms")


# ============================================================
# 3. ACTION SPACE + REWARD  (unchanged - identical to before)
# ============================================================
ACTIONS = ["allow", "rate_limit", "isolate", "block"]
IDEAL_ACTION = {"Normal Traffic":"allow", "Port Scanning":"rate_limit",
                "Brute Force":"isolate", "Web Attacks":"isolate", "Bots":"isolate",
                "DoS":"block", "DDoS":"block"}
SEVERITY = {"allow":0, "rate_limit":1, "isolate":2, "block":3}

def compute_reward(tc, a):
    ideal = IDEAL_ACTION[tc]
    if a == ideal: return 1.0
    if tc == "Normal Traffic" and a != "allow": return -1.0
    if tc != "Normal Traffic" and a == "allow": return -1.0
    return max(-0.5, 0.5 - 0.5*abs(SEVERITY[a]-SEVERITY[ideal]))


# ============================================================
# 4. LinUCB  (unchanged - alpha=1.0, single online pass)
# ============================================================
class LinUCB:
    def __init__(self, n_actions, d, alpha=1.0):
        self.n_actions=n_actions; self.alpha=alpha
        self.A=[np.identity(d) for _ in range(n_actions)]
        self.b=[np.zeros((d,1)) for _ in range(n_actions)]
    def select(self, x):
        x=x.reshape(-1,1); ba,bp=0,-1e18
        for a in range(self.n_actions):
            Ai=np.linalg.inv(self.A[a]); th=Ai@self.b[a]
            p=float((th.T@x).item())+self.alpha*float(np.sqrt((x.T@Ai@x).item()))
            if p>bp: bp,ba=p,a
        return ba
    def update(self,a,x,r): x=x.reshape(-1,1); self.A[a]+=x@x.T; self.b[a]+=r*x
    def act_greedy(self,x):
        x=x.reshape(-1,1); ba,bp=0,-1e18
        for a in range(self.n_actions):
            th=np.linalg.inv(self.A[a])@self.b[a]; p=float((th.T@x).item())
            if p>bp: bp,ba=p,a
        return ba


# ============================================================
# 5. 100-SEED BANDIT vs HONEST STATIC  (identical protocol)
# ============================================================
dominant_feat = np.abs(attribution).argmax(axis=1)

def run_once(seed):
    n = len(attribution)
    tr, te = safe_split(np.arange(n), true_classes,
                        test_size=0.30, random_state=seed)
    Xc = StandardScaler().fit_transform(attribution).astype(np.float64)

    bar = defaultdict(lambda: np.zeros(len(ACTIONS)))
    bac = defaultdict(lambda: np.zeros(len(ACTIONS)))
    for i in tr:
        f = dominant_feat[i]
        for ai, a in enumerate(ACTIONS):
            bar[f][ai] += compute_reward(true_classes[i], a); bac[f][ai] += 1
    smap = {f: int(np.argmax(bar[f]/np.maximum(bac[f],1))) for f in bar}

    bandit = LinUCB(len(ACTIONS), Xc.shape[1], alpha=1.0)
    rng = np.random.default_rng(seed)
    for i in rng.permutation(tr):
        a = bandit.select(Xc[i])
        bandit.update(a, Xc[i], compute_reward(true_classes[i], ACTIONS[a]))

    def ev(fn):
        rs, cor, fp, nn = [], 0, 0, 0
        for i in te:
            a = fn(i); rs.append(compute_reward(true_classes[i], a))
            cor += (a == IDEAL_ACTION[true_classes[i]])
            if true_classes[i]=="Normal Traffic":
                nn += 1; fp += (a!="allow")
        return cor/len(te), float(np.mean(rs)), fp/max(1,nn)

    s = ev(lambda i: ACTIONS[smap.get(dominant_feat[i], 1)])
    b = ev(lambda i: ACTIONS[bandit.act_greedy(Xc[i])])
    return dict(s_acc=s[0], s_rew=s[1], s_fp=s[2],
                b_acc=b[0], b_rew=b[1], b_fp=b[2])

print("\nRunning 100 seeds...")
res = pd.DataFrame([{**run_once(s), "seed":s} for s in range(100)])

def ci95(x):
    x=np.asarray(x); h=stats.sem(x)*stats.t.ppf(0.975, len(x)-1)
    return (x.mean()-h, x.mean()+h)
def fmt(c):
    lo,hi=ci95(res[c]); return f"{res[c].mean():.4f} ± {res[c].std():.4f} (95% CI [{lo:.4f},{hi:.4f}])"

wins  = int((res["b_rew"]>res["s_rew"]).sum())
w_rew = stats.wilcoxon(res["b_rew"], res["s_rew"])
w_acc = stats.wilcoxon(res["b_acc"], res["s_acc"])

print("\n"+"="*74)
print("InSDN (XGBoost) - MITIGATION: bandit vs honest static (100 seeds)")
print("="*74)
for m,label in [("acc","ACCURACY"),("rew","REWARD"),("fp","FP_RATE")]:
    print(f"\n{label}")
    print(f"  static: {fmt('s_'+m)}")
    print(f"  bandit: {fmt('b_'+m)}")
print("\n"+"="*74)
print(f"Bandit beats static on reward: {wins}/100 seeds")
print(f"Wilcoxon p(reward)={w_rew.pvalue:.2e} | p(accuracy)={w_acc.pvalue:.2e}")
print("="*74)


# ============================================================
# 6. SAVE  (xgb_* names - RF artifacts are left untouched)
# ============================================================
attribution_df = pd.DataFrame(attribution,
                              columns=[f"shap_{c}" for c in X_explain.columns])
attribution_df["predicted_class"] = [le.classes_[c] for c in pred]
attribution_df["true_class"] = true_classes

joblib.dump(xgb, "/content/artifacts/xgb_insdn.pkl")
attribution_df.to_parquet("/content/artifacts/shap_attributions_xgb_insdn.parquet")
res.to_csv("/content/artifacts/phase4_multiseed100_xgb_insdn.csv", index=False)

log_lines = [
 f"[InSDN/XGB] Detection | acc {acc:.4f} macroF1 {f1:.4f} | train {train_s:.1f}s",
 f"[InSDN/XGB] SHAP latency | mean {lat['mean']:.2f} median {lat['median']:.2f} "
 f"p95 {lat['p95']:.2f} ms | {EXPLAIN_N} flows",
 f"[InSDN/XGB] Mitigation 100-seed | STATIC acc {res['s_acc'].mean():.4f}"
 f"±{res['s_acc'].std():.4f} rew {res['s_rew'].mean():.4f}±{res['s_rew'].std():.4f} || "
 f"BANDIT acc {res['b_acc'].mean():.4f}±{res['b_acc'].std():.4f} "
 f"rew {res['b_rew'].mean():.4f}±{res['b_rew'].std():.4f} | wins {wins}/100 | "
 f"Wilcoxon p(rew)={w_rew.pvalue:.2e}",
]
with open("/content/artifacts/results_log.txt","a") as fh:
    fh.write("\n".join(log_lines)+"\n")

try:
    os.makedirs(DRIVE_DIR, exist_ok=True)
    joblib.dump(xgb, f"{DRIVE_DIR}/xgb_insdn.pkl")
    attribution_df.to_parquet(f"{DRIVE_DIR}/shap_attributions_xgb_insdn.parquet")
    res.to_csv(f"{DRIVE_DIR}/phase4_multiseed100_xgb_insdn.csv", index=False)
    with open(f"{DRIVE_DIR}/results_log.txt","a") as fh:
        fh.write("\n".join(log_lines)+"\n")
    print("\nSaved XGB artifacts to /content/artifacts and Drive.")
except Exception as e:
    print("\nSaved locally. (Drive copy skipped:", e, ")")

print("\nPASTE INTO RESULTS LOG:")
for l in log_lines: print(" ", l)
print("""
COMPARE AGAINST THE InSDN RF RESULT BEFORE COMMITTING TO THE SWAP:
  RF  : detection F1 0.9910 | bandit 100/100 | p(rew)=3.88e-18
        (static acc 0.7827 vs bandit 0.9952 - a 21-point gap)
  XGB : see above.
Watch the static baseline specifically. On CICIDS2017 the swap lifted static
from 0.9424 to 0.9668 and shrank the gap from 3.4 to 0.7 points, because
XGBoost's shallower trees concentrate attribution signal into fewer features.
If the same happens here, the 21-point InSDN gap will narrow too - report it.
If the bandit still wins clearly, the swap is safe. If the margin collapsed,
that is a real finding and RF may be the better pipeline choice - report
whichever you measure, do not pick the prettier one.
""")

XGBoost swap on 150,000 train / 50,000 test, 79 features, 5 classes.

DETECTION - InSDN (XGBoost)
Accuracy 0.9999 | Macro-F1 0.9986 | trained in 14.9s
                precision    recall  f1-score   support

   Brute Force       0.99      1.00      0.99        72
          DDoS       1.00      1.00      1.00     17921
           DoS       1.00      1.00      1.00       279
Normal Traffic       1.00      1.00      1.00     16676
 Port Scanning       1.00      1.00      1.00     15052

      accuracy                           1.00     50000
     macro avg       1.00      1.00      1.00     50000
  weighted avg       1.00      1.00      1.00     50000


SHAP batch over 3000 flows: 0.6s (0.19 ms/flow amortized) | shape (3000, 79, 5)
SHAP per-flow latency: mean 7.74 ms | median 7.47 ms | p95 9.21 ms

Running 100 seeds...

InSDN (XGBoost) - MITIGATION: bandit vs honest static (100 seeds)

ACCURACY
  static: 0.9943 ± 0.0007 (95% CI [0.9942,0.9945])
  bandit: 0.9970 ± 0.0015 (95% CI [0.9967,0.

In [ ]:
# ============================================================
#  FULL-DATA DETECTORS  —  retrain RF (and XGBoost if needed) on ALL rows
#
#  You already have XGBoost on the full set from the swap cell. This cell
#  brings Random Forest up to the same footing, reusing the hyperparameters
#  of the RF already in memory so the ONLY thing that changes is the number
#  of training rows.
#
#  Afterwards you will have a matched full-data trio:
#      rf_full   XGBoost (your `xgb`)   DNN (dnn_shap.py with TRAIN_N = None)
#
#  Paste as ONE cell after Phase 2. Needs X_train, y_train, and an existing
#  `rf` to copy settings from (any RF - its training size does not matter,
#  only its hyperparameters are read).
#
#  Runtime: a few minutes. RF training is parallel; the SINGLE-THREAD pinning
#  that matters for fairness happens later, in resource_benchmark.py.
# ============================================================

RF_VAR   = "rf"       # existing RF whose hyperparameters to copy
OUT_NAME = "rf_full"  # new variable this cell creates
# ============================================================

import numpy as np, time
from sklearn.ensemble import RandomForestClassifier

for _v in ["X_train", "y_train", RF_VAR]:
    assert _v in globals(), f"{_v} missing."

_src = globals()[RF_VAR]
_params = _src.get_params()

def _root_rows(m):
    try:
        return int(round(m.estimators_[0].tree_.n_node_samples[0] / 0.632))
    except Exception:
        return None

print(f"copying hyperparameters from `{RF_VAR}` "
      f"(fitted on ~{_root_rows(_src):,} rows)")
print(f"  n_estimators={_params.get('n_estimators')} "
      f"max_depth={_params.get('max_depth')} "
      f"min_samples_leaf={_params.get('min_samples_leaf')} "
      f"class_weight={_params.get('class_weight')}")
print(f"\ntraining on the FULL set: {len(X_train):,} rows x "
      f"{X_train.shape[1]} features")

_params["n_jobs"] = -1          # fast to train; threads are pinned later
t0 = time.perf_counter()
rf_full = RandomForestClassifier(**_params).fit(X_train, y_train)
print(f"done in {time.perf_counter() - t0:.1f}s")

globals()[OUT_NAME] = rf_full
print(f"\n`{OUT_NAME}` fitted on ~{_root_rows(rf_full):,} rows "
      f"(root-node estimate)")

# ---- confirm the full-data XGBoost is the one you think it is ----
print("\nXGBoost models in memory (larger root cover = more training rows):")
for k, v in sorted(globals().items()):
    if k.startswith("_"):
        continue
    if type(v).__name__.startswith("XGB") and hasattr(v, "get_booster"):
        try:
            df = v.get_booster().trees_to_dataframe()
            print(f"  {k:12s} root cover {df[df.Node == 0].Cover.mean():>12,.0f}")
        except Exception as e:
            print(f"  {k:12s} (cover unavailable: {type(e).__name__})")

print(f"""
NEXT
  1. dnn_shap.py          with TRAIN_N = None   (full set, GPU-trained if
                          a GPU is attached; measured on CPU either way)
  2. resource_benchmark.py with
         BENCH_RF  = "{OUT_NAME}"
         BENCH_XGB = "<the LARGER-cover XGBoost above>"
         BENCH_EBM = None
     It prints each model's inferred training size and warns if they differ.
  3. full_metrics.py

  The 60k models stay in memory under their old names, so you keep both
  sets of numbers. Quote one or the other - never a row from each.
""")

copying hyperparameters from `rf` (fitted on ~149,840 rows)
  n_estimators=100 max_depth=None min_samples_leaf=1 class_weight=balanced

training on the FULL set: 150,000 rows x 79 features
done in 13.0s

`rf_full` fitted on ~149,840 rows (root-node estimate)

XGBoost models in memory (larger root cover = more training rows):
  xgb          root cover        2,875

NEXT
  1. dnn_shap.py          with TRAIN_N = None   (full set, GPU-trained if
                          a GPU is attached; measured on CPU either way)
  2. resource_benchmark.py with
         BENCH_RF  = "rf_full"
         BENCH_XGB = "<the LARGER-cover XGBoost above>"
         BENCH_EBM = None
     It prints each model's inferred training size and warns if they differ.
  3. full_metrics.py

  The 60k models stay in memory under their old names, so you keep both
  sets of numbers. Quote one or the other - never a row from each.



In [ ]:
# ============================================================
#  DEPTH CONTROL + DETECTION METRICS
#
#  Does two things in one cell:
#    1. trains a depth-matched Random Forest (same hyperparameters as the
#       full one, only max_depth changed) - the control that answers
#       "isn't the explanation-cost gap just tree depth?"
#    2. prints accuracy / macro-P / macro-R / macro-F1 for every detector,
#       so the detection table and the resource table describe the SAME
#       models - no name-based guessing.
#
#  Run AFTER full_data_models.py and the XGBoost swap cell.
#  Needs in memory: X_train, y_train, X_test, y_test, rf_full, and an XGBoost.
#
#  Runtime: 2-5 min (the depth-capped forest trains quickly).
# ============================================================

MATCH_DEPTH = 6        # set to the XGBoost max_depth; None = read it from XGB
RF_SOURCE   = "rf_full"
OUT_NAME    = "rf_depth6"
# ============================================================

import time, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)

for _v in ["X_train", "y_train", "X_test", "y_test", RF_SOURCE]:
    assert _v in globals(), f"{_v} missing."

_src = globals()[RF_SOURCE]

# ---- find the XGBoost by type, and read its depth ----
_xgs = [(k, v) for k, v in list(globals().items())
        if not k.startswith("_") and type(v).__name__.startswith("XGB")
        and hasattr(v, "feature_importances_")]
if len(_xgs) > 1:
    print("NOTE: several XGBoost models in memory -", ", ".join(k for k, _ in _xgs))
    print("      root cover (larger = more training rows):")
    for k, v in _xgs:
        try:
            df = v.get_booster().trees_to_dataframe()
            print(f"        {k:12s} {df[df.Node == 0].Cover.mean():>12,.0f}")
        except Exception:
            print(f"        {k:12s} (unavailable)")
_xgb_name, _xgb = (_xgs[0] if _xgs else (None, None))

if MATCH_DEPTH is None:
    MATCH_DEPTH = (_xgb.get_params().get("max_depth") if _xgb is not None else 6)
    print(f"matching XGBoost's max_depth = {MATCH_DEPTH}")

def _geom(m):
    t = m.estimators_[0].tree_
    rows = int(round(t.n_node_samples[0] / 0.632))   # bootstrap -> ~63.2% distinct
    return t.max_depth, t.n_leaves, rows

d, l, n = _geom(_src)
print(f"{RF_SOURCE}: depth {d}, {l:,} leaves/tree, ~{n:,} training rows")

# ---- train the control ----
p = _src.get_params()
p["max_depth"] = MATCH_DEPTH
p["n_jobs"] = -1                 # fast to fit; threads get pinned at benchmark time
print(f"\ntraining {OUT_NAME} at max_depth={MATCH_DEPTH} on {len(X_train):,} rows ...")
t0 = time.perf_counter()
_ctrl = RandomForestClassifier(**p).fit(X_train, y_train)
globals()[OUT_NAME] = _ctrl
print(f"  done in {time.perf_counter() - t0:.1f}s")

d2, l2, n2 = _geom(_ctrl)
print(f"{OUT_NAME}: depth {d2}, {l2:,} leaves/tree, ~{n2:,} training rows")
print(f"\nleaves per tree: {l:,} -> {l2:,}  ({l / max(l2, 1):.0f}x fewer)")

# ---- detection metrics for every detector, same test set ----
targets = [(RF_SOURCE, _src), (OUT_NAME, _ctrl)]
if _xgb is not None:
    targets.append((_xgb_name, _xgb))

print("\n" + "=" * 78)
print("DETECTION QUALITY (macro metrics - the honest view under imbalance)")
print("=" * 78)
print(f"{'model':14s} {'acc':>8s} {'macro-P':>9s} {'macro-R':>9s} {'macro-F1':>9s}")
rows = []
for nm, m in targets:
    yp = m.predict(X_test)
    r = dict(model=nm,
             acc=accuracy_score(y_test, yp),
             P=precision_score(y_test, yp, average="macro", zero_division=0),
             R=recall_score(y_test, yp, average="macro", zero_division=0),
             F1=f1_score(y_test, yp, average="macro", zero_division=0))
    rows.append((r, yp))
    print(f"{nm:14s} {r['acc']:8.4f} {r['P']:9.4f} {r['R']:9.4f} {r['F1']:9.4f}")
print("=" * 78)

if "le" in globals():
    print("\nPer-class detail (the macro gap usually lives in one or two "
          "small classes):")
    for r, yp in rows:
        print("\n" + "-" * 70); print(r["model"]); print("-" * 70)
        print(classification_report(y_test, yp,
                                    labels=np.arange(len(le.classes_)),
                                    target_names=list(le.classes_),
                                    zero_division=0))

print(f"""
NEXT - resource_benchmark.py with:

    BENCH_RF    = "{RF_SOURCE}"
    BENCH_XGB   = "{_xgb_name}"
    BENCH_EBM   = None
    BENCH_EXTRA = {{"Random Forest (depth {MATCH_DEPTH}) + SHAP": "{OUT_NAME}"}}

  Make sure there is only ONE BENCH_EXTRA line in that cell, and that it
  prints "benchmarking 4 detector(s)" before the rounds begin.
""")

log = "[DEPTH-CONTROL] " + " | ".join(
    f"{r['model']}: acc {r['acc']:.4f} P {r['P']:.4f} R {r['R']:.4f} "
    f"F1 {r['F1']:.4f}" for r, _ in rows)
import os
for _d in ["/content/artifacts", "/content/drive/MyDrive/CN_PROJECT/artifacts"]:
    try:
        os.makedirs(_d, exist_ok=True)
        with open(f"{_d}/results_log.txt", "a") as fh:
            fh.write(log + "\n")
    except Exception:
        pass
print("PASTE INTO RESULTS LOG:\n ", log)

rf_full: depth 22, 124 leaves/tree, ~149,840 training rows

training rf_depth6 at max_depth=6 on 150,000 rows ...
  done in 16.0s
rf_depth6: depth 6, 26 leaves/tree, ~149,840 training rows

leaves per tree: 124 -> 26  (5x fewer)

DETECTION QUALITY (macro metrics - the honest view under imbalance)
model               acc   macro-P   macro-R  macro-F1
rf_full          0.9998    0.9963    0.9861    0.9910
rf_depth6        0.9450    0.7188    0.9611    0.7404
xgb              0.9999    0.9972    0.9999    0.9986

Per-class detail (the macro gap usually lives in one or two small classes):

----------------------------------------------------------------------
rf_full
----------------------------------------------------------------------
                precision    recall  f1-score   support

   Brute Force       0.99      0.93      0.96        72
          DDoS       1.00      1.00      1.00     17921
           DoS       1.00      1.00      1.00       279
Normal Traffic       1.00      1.

In [ ]:
# ============================================================
#  DNN + SHAP  —  fourth detector for the algorithm comparison
#
#  Adds a feed-forward neural network to the RF / XGBoost / EBM
#  comparison, and measures what SHAP actually costs on it:
#    * DeepSHAP   (shap.DeepExplainer)   - the realistic deployment choice
#    * KernelSHAP (shap.KernelExplainer) - the model-agnostic worst case
#
#  Paste as ONE cell in the SAME session that ran Phase 1-2, so
#  X_train / X_test / y_train / y_test / le are already in memory.
#
#  FAIRNESS: every model in the comparison must be single-threaded or the
#  latency numbers are an artifact of thread count, not of the model.
#  This cell pins torch to 1 thread. Make sure the RF/XGB cell used
#  n_jobs=1 / nthread=1 before you quote the numbers side by side.
#
#  Runtime: ~3-8 min on Colab CPU (KernelSHAP is the slow part).
# ============================================================

DATASET      = "InSDN"   # label only - for the printout and filenames
TRAIN_N      = None           # None = FULL training set; or an int to subsample.
                              # Must match what RF and XGBoost were trained on.
TRAIN_DEVICE = "auto"         # "auto" uses GPU for TRAINING only if present.
                              # Inference and SHAP are ALWAYS measured on CPU,
                              # single-threaded, so they stay comparable with
                              # the tree models. Training time is not the
                              # metric being compared - explanation cost is -
                              # so a GPU fit is legitimate. Say so in methods.
EPOCHS       = 30
BATCH        = 512
HIDDEN       = (128, 64)
DROPOUT      = 0.2
LR           = 1e-3
SEED         = 42

N_EXPLAIN_DEEP   = 500        # rows timed with DeepSHAP
N_EXPLAIN_KERNEL = 10         # rows timed with KernelSHAP (it is very slow)
KERNEL_BG        = 50         # background rows for KernelSHAP (kmeans summary)
DEEP_BG          = 100        # background rows for DeepSHAP
# ============================================================

import os, time, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

for _v in ["X_train", "X_test", "y_train", "y_test", "le"]:
    assert _v in globals(), f"{_v} missing - re-run the Phase 1-2 cells first."

try:
    import torch, torch.nn as nn
except ImportError:
    raise SystemExit("PyTorch not available. On Colab it is pre-installed; "
                     "otherwise: pip install torch")
import shap
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)

# --- single thread for MEASUREMENT, so the comparison is honest ---
torch.set_num_threads(1)
os.environ.setdefault("OMP_NUM_THREADS", "1")
torch.manual_seed(SEED); np.random.seed(SEED)

# Measurement is always CPU. Training may use a GPU when one is present.
MEASURE_DEVICE = torch.device("cpu")
if TRAIN_DEVICE == "auto":
    TRAIN_DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    TRAIN_DEV = torch.device(TRAIN_DEVICE)
print(f"training on {TRAIN_DEV.type}, measuring on cpu (1 thread)")

# ------------------------------------------------------------------
# 1. data prep  (a DNN needs scaling; the trees did not)
# ------------------------------------------------------------------
# copy=True: np.asarray on a pandas frame can hand back a read-only view under
# copy-on-write, and the in-place nan_to_num below then fails.
Xtr_raw = np.array(X_train, dtype=np.float64, copy=True)
Xte_raw = np.array(X_test,  dtype=np.float64, copy=True)
ytr     = np.asarray(y_train).astype(np.int64)
yte     = np.asarray(y_test).astype(np.int64)

# CICIDS2017 carries +/-inf in a few rate columns; clip before scaling
for A in (Xtr_raw, Xte_raw):
    np.nan_to_num(A, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

if TRAIN_N and TRAIN_N < len(Xtr_raw):
    rs  = np.random.RandomState(SEED)
    idx = rs.choice(len(Xtr_raw), TRAIN_N, replace=False)
    Xtr_raw, ytr = Xtr_raw[idx], ytr[idx]
    print(f"Subsampled train to {TRAIN_N:,} rows (matching the other detectors)")
else:
    print(f"Training on the FULL set: {len(Xtr_raw):,} rows "
          f"- RF and XGBoost must be trained on these same rows")
N_TRAIN_USED = len(Xtr_raw)

scaler = StandardScaler().fit(Xtr_raw)
Xtr = scaler.transform(Xtr_raw).astype(np.float32)
Xte = scaler.transform(Xte_raw).astype(np.float32)

N_FEAT    = Xtr.shape[1]
N_CLASSES = len(le.classes_)
print(f"{DATASET}: {Xtr.shape[0]:,} train / {Xte.shape[0]:,} test | "
      f"{N_FEAT} features | {N_CLASSES} classes")

# class weights - the rare classes are the whole macro-F1 story
counts  = np.bincount(ytr, minlength=N_CLASSES).astype(np.float64)
weights = np.where(counts > 0, counts.sum() / (N_CLASSES * np.maximum(counts, 1)), 0.0)
print("class weights:", np.round(weights, 3))

# ------------------------------------------------------------------
# 2. the model  (plain Linear/ReLU/Dropout - DeepExplainer handles these
#    reliably; BatchNorm is a common source of DeepSHAP failures)
# ------------------------------------------------------------------
def build():
    layers, prev = [], N_FEAT
    for h in HIDDEN:
        layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(DROPOUT)]
        prev = h
    layers += [nn.Linear(prev, N_CLASSES)]
    return nn.Sequential(*layers)

model = build().to(TRAIN_DEV)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
lossf = nn.CrossEntropyLoss(
    weight=torch.tensor(weights, dtype=torch.float32).to(TRAIN_DEV))

# 10% validation split for early stopping
rs      = np.random.RandomState(SEED)
perm    = rs.permutation(len(Xtr))
n_val   = max(1, int(0.1 * len(Xtr)))
va_i, tr_i = perm[:n_val], perm[n_val:]
Xt = torch.from_numpy(Xtr[tr_i]).to(TRAIN_DEV)
yt = torch.from_numpy(ytr[tr_i]).to(TRAIN_DEV)
Xv = torch.from_numpy(Xtr[va_i]).to(TRAIN_DEV)
yv = torch.from_numpy(ytr[va_i]).to(TRAIN_DEV)
print(f"  {len(Xt):,} train / {len(Xv):,} val rows on {TRAIN_DEV.type}")

best_val, best_state, patience, bad = float("inf"), None, 5, 0
t_train0 = time.perf_counter(); c_train0 = time.process_time()

for ep in range(EPOCHS):
    model.train()
    # index tensor must live on the same device as the data it indexes
    order = torch.randperm(len(Xt), device=TRAIN_DEV)
    for i in range(0, len(Xt), BATCH):
        b = order[i:i + BATCH]
        opt.zero_grad()
        loss = lossf(model(Xt[b]), yt[b])
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():                      # chunked: val set can be large
        tot, n = 0.0, 0
        for i in range(0, len(Xv), 8192):
            xb, yb = Xv[i:i + 8192], yv[i:i + 8192]
            tot += lossf(model(xb), yb).item() * len(xb); n += len(xb)
        vl = tot / max(n, 1)
    if vl < best_val - 1e-5:
        best_val, bad = vl, 0
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= patience:
            print(f"  early stop at epoch {ep + 1}")
            break
    if (ep + 1) % 5 == 0:
        print(f"  epoch {ep + 1:3d}  val loss {vl:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

if TRAIN_DEV.type == "cuda":
    torch.cuda.synchronize()
train_wall = time.perf_counter() - t_train0
train_cpu  = time.process_time() - c_train0

# EVERYTHING below this line is measured on CPU, single-threaded, so the DNN
# is timed on the same footing as the tree models. Free the GPU copies first.
model = model.to(MEASURE_DEVICE)
model.eval()
del Xt, yt, Xv, yv
if TRAIN_DEV.type == "cuda":
    torch.cuda.empty_cache()
torch.set_num_threads(1)

print(f"\nTrained in {train_wall:.1f}s wall / {train_cpu:.1f}s CPU "
      f"on {TRAIN_DEV.type}; model moved to cpu for all measurements")

# ------------------------------------------------------------------
# 3. detection quality
# ------------------------------------------------------------------
def predict(Xnp, bs=4096):
    out = []
    with torch.no_grad():
        for i in range(0, len(Xnp), bs):
            out.append(model(torch.from_numpy(Xnp[i:i + bs])).argmax(1).numpy())
    return np.concatenate(out)

yp = predict(Xte)
row = {
    "Model":           "DNN + SHAP",
    "Accuracy":        round(accuracy_score(yte, yp), 4),
    "Macro-Precision": round(precision_score(yte, yp, average="macro", zero_division=0), 4),
    "Macro-Recall":    round(recall_score(yte, yp, average="macro", zero_division=0), 4),
    "Macro-F1":        round(f1_score(yte, yp, average="macro", zero_division=0), 4),
    "Wtd-F1":          round(f1_score(yte, yp, average="weighted", zero_division=0), 4),
}
print(f"\n  acc {row['Accuracy']:.4f} | P {row['Macro-Precision']:.4f} | "
      f"R {row['Macro-Recall']:.4f} | F1 {row['Macro-F1']:.4f}")

# ------------------------------------------------------------------
# 4. inference latency - batched AND batch-1 (the online case)
# ------------------------------------------------------------------
rs2   = np.random.RandomState(SEED)
e_idx = rs2.choice(len(Xte), min(N_EXPLAIN_DEEP, len(Xte)), replace=False)
Xe    = Xte[e_idx]
Xe_t  = torch.from_numpy(Xe)

with torch.no_grad():                      # warm up
    model(Xe_t[:32])

def timed(fn, reps=3):
    """median wall and cpu seconds over reps"""
    ws, cs = [], []
    for _ in range(reps):
        w0, c0 = time.perf_counter(), time.process_time()
        fn()
        ws.append(time.perf_counter() - w0); cs.append(time.process_time() - c0)
    return float(np.median(ws)), float(np.median(cs))

inf_w, inf_c = timed(lambda: model(Xe_t).argmax(1))
inf_ms     = 1000 * inf_w / len(Xe)
inf_cpu_ms = 1000 * inf_c / len(Xe)

n1 = min(200, len(Xe))
def _b1():
    with torch.no_grad():
        for r in range(n1):
            model(Xe_t[r:r + 1])
b1_w, b1_c = timed(_b1, reps=2)
b1_ms = 1000 * b1_w / n1

print(f"\ninference  batched {inf_ms:.4f} ms/flow (CPU {inf_cpu_ms:.4f}) | "
      f"batch-1 {b1_ms:.4f} ms/flow")

# ------------------------------------------------------------------
# 5. DeepSHAP
# ------------------------------------------------------------------
bg_idx = np.random.RandomState(SEED).choice(len(Xtr), min(DEEP_BG, len(Xtr)),
                                            replace=False)
bg_t   = torch.from_numpy(Xtr[bg_idx])

def _norm_sv(sv, n_rows):
    """shap returns list-per-class (old) or (rows, feats, classes) (new)."""
    if isinstance(sv, list):
        sv = np.stack(sv, axis=-1)
    sv = np.asarray(sv)
    if sv.ndim == 2:
        sv = sv[:, :, None]
    assert sv.shape[0] == n_rows and sv.shape[1] == N_FEAT, \
        f"unexpected SHAP shape {sv.shape} for {n_rows} rows x {N_FEAT} feats"
    return sv

deep_ms = deep_cpu_ms = float("nan"); deep_sv = None; deep_kind = "DeepExplainer"
try:
    de = shap.DeepExplainer(model, bg_t)
    de.shap_values(Xe_t[:8], check_additivity=False)          # warm up
    w0, c0 = time.perf_counter(), time.process_time()
    deep_sv = de.shap_values(Xe_t, check_additivity=False)
    dw, dc = time.perf_counter() - w0, time.process_time() - c0
    deep_ms, deep_cpu_ms = 1000 * dw / len(Xe), 1000 * dc / len(Xe)
    deep_sv = _norm_sv(deep_sv, len(Xe))
    print(f"DeepSHAP   {deep_ms:.4f} ms/flow (CPU {deep_cpu_ms:.4f}) "
          f"over {len(Xe)} flows   shape {deep_sv.shape}")
except Exception as e:
    print(f"DeepExplainer failed ({type(e).__name__}: {e})\n  -> GradientExplainer")
    try:
        ge = shap.GradientExplainer(model, bg_t)
        ge.shap_values(Xe_t[:8])
        w0, c0 = time.perf_counter(), time.process_time()
        deep_sv = ge.shap_values(Xe_t)
        dw, dc = time.perf_counter() - w0, time.process_time() - c0
        deep_ms, deep_cpu_ms = 1000 * dw / len(Xe), 1000 * dc / len(Xe)
        deep_sv = _norm_sv(deep_sv, len(Xe))
        deep_kind = "GradientExplainer"
        print(f"GradientSHAP {deep_ms:.4f} ms/flow (CPU {deep_cpu_ms:.4f}) "
              f"shape {deep_sv.shape}")
    except Exception as e2:
        print(f"GradientExplainer also failed ({type(e2).__name__}: {e2}) "
              f"- report DeepSHAP as unavailable, do not substitute a number")
        deep_kind = "unavailable"

# ------------------------------------------------------------------
# 6. KernelSHAP - the model-agnostic worst case
#    Timed on a handful of rows; the per-flow cost is what you report.
# ------------------------------------------------------------------
kern_ms = kern_cpu_ms = float("nan")
try:
    def f_np(A):
        with torch.no_grad():
            return torch.softmax(model(torch.from_numpy(A.astype(np.float32))),
                                 dim=1).numpy()
    bgk = shap.kmeans(Xtr[bg_idx], min(KERNEL_BG, len(bg_idx)))
    ke  = shap.KernelExplainer(f_np, bgk)
    nk  = min(N_EXPLAIN_KERNEL, len(Xe))
    w0, c0 = time.perf_counter(), time.process_time()
    _ = ke.shap_values(Xe[:nk], silent=True)
    kw, kc = time.perf_counter() - w0, time.process_time() - c0
    kern_ms, kern_cpu_ms = 1000 * kw / nk, 1000 * kc / nk
    print(f"KernelSHAP {kern_ms:.1f} ms/flow (CPU {kern_cpu_ms:.1f}) "
          f"measured over {nk} flows")
except Exception as e:
    print(f"KernelSHAP failed ({type(e).__name__}: {e})")

# ------------------------------------------------------------------
# 7. model size on disk
# ------------------------------------------------------------------
tmp = "/content/_dnn_tmp.pt" if os.path.isdir("/content") else "/tmp/_dnn_tmp.pt"
torch.save(model.state_dict(), tmp)
size_kb = os.path.getsize(tmp) / 1024
os.remove(tmp)
n_params = sum(p.numel() for p in model.parameters())
print(f"model      {n_params:,} params | {size_kb:.1f} KB on disk")

# ------------------------------------------------------------------
# 8. summary + persistence
# ------------------------------------------------------------------
row.update({
    "Inference (ms)":   round(inf_ms, 4),
    "Batch-1 (ms)":     round(b1_ms, 4),
    "DeepSHAP (ms)":    None if np.isnan(deep_ms) else round(deep_ms, 4),
    "KernelSHAP (ms)":  None if np.isnan(kern_ms) else round(kern_ms, 2),
    "End-to-end (ms)":  None if np.isnan(deep_ms) else round(inf_ms + deep_ms, 4),
    "Flows/sec":        None if np.isnan(deep_ms) else round(1000 / (inf_ms + deep_ms), 1),
    "Params":           n_params,
    "Size (KB)":        round(size_kb, 1),
    "Train wall (s)":   round(train_wall, 1),
    "Train CPU (s)":    round(train_cpu, 1),
    "Explainer":        deep_kind,
})

print("\n" + "=" * 92)
print(f"{DATASET} - DNN + SHAP")
print("=" * 92)
for k, v in row.items():
    print(f"  {k:18s} {v}")
print("=" * 92)

print("\nPer-class detail:")
print(classification_report(yte, yp, labels=np.arange(N_CLASSES),
                            target_names=list(le.classes_), zero_division=0))

TAG  = DATASET.lower().replace(" ", "")
DIRS = ["/content/artifacts", "/content/drive/MyDrive/CN_PROJECT/artifacts"]
for d in DIRS:
    try:
        os.makedirs(d, exist_ok=True)
        pd.DataFrame([row]).to_csv(f"{d}/dnn_shap_{TAG}.csv", index=False)
        if deep_sv is not None:
            # Same column convention as the other attribution files, so the
            # bandit / reward-ablation cells can read this one unchanged:
            #   shap_<feature> ... , true_class (string label), pred_class
            assert deep_sv.shape[2] == N_CLASSES, (
                f"SHAP returned {deep_sv.shape[2]} class slices but there are "
                f"{N_CLASSES} classes - do not save this, investigate first")
            cols = (list(X_test.columns) if hasattr(X_test, "columns")
                    else [f"f{i}" for i in range(N_FEAT)])
            pred_e = predict(Xe)
            A = np.stack([deep_sv[i, :, pred_e[i]] for i in range(len(Xe))])
            att = pd.DataFrame(A, columns=[f"shap_{c}" for c in cols])
            att["true_class"] = le.inverse_transform(yte[e_idx])
            att["pred_class"] = le.inverse_transform(pred_e)
            att.to_parquet(f"{d}/shap_attributions_dnn_{TAG}.parquet", index=False)
    except Exception as e:
        print(f"(skipped {d}: {e})")

log = (f"[DNN/{DATASET}] acc {row['Accuracy']:.4f} P {row['Macro-Precision']:.4f} "
       f"R {row['Macro-Recall']:.4f} F1 {row['Macro-F1']:.4f} | "
       f"inf {inf_ms:.3f}ms batch1 {b1_ms:.3f}ms | {deep_kind} {deep_ms:.3f}ms | "
       f"kernel {kern_ms:.1f}ms | {n_params:,} params {size_kb:.1f}KB")
for d in DIRS:
    try:
        with open(f"{d}/results_log.txt", "a") as fh:
            fh.write(log + "\n")
    except Exception:
        pass
print("\nPASTE INTO RESULTS LOG:\n ", log)

# Sentinel for resource_benchmark.py: proves THIS cell produced the `model`
# and `scaler` now in memory. Without it the benchmark refuses to touch them,
# so a stale pair left by some other cell can never be measured by mistake.
DNN_READY = {"dataset": DATASET, "n_feat": N_FEAT, "n_classes": N_CLASSES,
             "deep_bg": DEEP_BG, "explainer": deep_kind,
             "n_train": N_TRAIN_USED, "train_device": TRAIN_DEV.type}

print("""
READING THESE NUMBERS
  DeepSHAP is the number to put in the comparison table - it is what you would
  actually deploy. KernelSHAP is the model-agnostic fallback and is typically
  orders of magnitude slower; report it as the cost of explaining a model that
  has no structure-specific explainer, which is the general case for DNNs.

  Quote CPU time as well as wall time. A tree ensemble with n_jobs=-1 hides its
  real cost behind parallelism; this cell is pinned to one thread, so if the
  RF/XGB numbers were taken multi-threaded the comparison is not valid.
""")

training on cuda, measuring on cpu (1 thread)
Training on the FULL set: 150,000 rows - RF and XGBoost must be trained on these same rows
InSDN: 150,000 train / 50,000 test | 79 features | 5 classes
class weights: [138.889   0.558  35.842   0.6     0.664]
  135,000 train / 15,000 val rows on cuda
  epoch   5  val loss 0.0200
  epoch  10  val loss 0.0076
  epoch  15  val loss 0.0048
  epoch  20  val loss 0.0044
  epoch  25  val loss 0.0042
  epoch  30  val loss 0.0023

Trained in 13.6s wall / 12.9s CPU on cuda; model moved to cpu for all measurements

  acc 0.9995 | P 0.9856 | R 0.9997 | F1 0.9925

inference  batched 0.0009 ms/flow (CPU 0.0010) | batch-1 0.0697 ms/flow
DeepSHAP   22.3531 ms/flow (CPU 22.3356) over 500 flows   shape (500, 79, 5)
KernelSHAP 214.6 ms/flow (CPU 214.6) measured over 10 flows
model      18,821 params | 76.5 KB on disk

InSDN - DNN + SHAP
  Model              DNN + SHAP
  Accuracy           0.9995
  Macro-Precision    0.9856
  Macro-Recall       0.9997
  Macro-

In [ ]:
n = len(y_train)
for i, c in enumerate(le.classes_):
    print(f"{c:20s} {(y_train==i).sum()/n:.4f}")

Brute Force          0.0014
DDoS                 0.3584
DoS                  0.0056
Normal Traffic       0.3335
Port Scanning        0.3010


In [ ]:
# ============================================================
#  RESOURCE BENCHMARK  —  what each detector actually costs
#
#  Measures, for every detector still in memory:
#    * inference      wall ms/flow AND cpu ms/flow
#    * explanation    wall ms/flow AND cpu ms/flow
#    * batch-1 latency (the online case a live IDS actually runs in)
#    * peak resident memory during the explanation call
#    * serialized model size on disk
#
#  INTERLEAVED: every model is measured once per round, round-robin, for
#  ROUNDS rounds. If Colab throttles mid-run it throttles all of them, so
#  the ordering survives. Medians reported with the full spread.
#
#  THREADS ARE PINNED TO 1 for every model. A tree ensemble with n_jobs=-1
#  hides its real cost behind parallelism; CPU-seconds is what you pay on a
#  shared or edge box. This cell re-pins what it can and WARNS about what it
#  cannot, so you know whether the table is quotable.
#
#  Paste as ONE cell in the SAME session as the comparison + DNN cells.
#  Nothing is retrained.
#
#  Runtime: ~2-6 min depending on how many detectors are loaded.
# ============================================================

DATASET   = "InSDN"   # label only
N_EXPLAIN = 500            # flows per explanation measurement
N_BATCH1  = 200            # flows timed one-at-a-time
ROUNDS    = 3              # interleaved repetitions

# WHICH models to benchmark. Leave as None to auto-detect by type.
# Set them explicitly when more than one model of a type is in memory - for
# example after running BOTH the full-data swap cell AND the glass-box cell,
# which each train their own RF and XGBoost on DIFFERENT numbers of rows.
# Models trained on different amounts of data are NOT comparable on latency,
# so the cell refuses to pick for you. Name the variables from the SAME cell:
#   BENCH_RF  = "rf_60k"
#   BENCH_XGB = "xgb_60k"
BENCH_RF    = "rf_full"
BENCH_XGB   = "xgb"
BENCH_EBM   = None
BENCH_EXTRA = {"Random Forest (depth 6) + SHAP": "rf_depth6"}       # name of the dict of EBMs, e.g. "ebms"

# Extra tree models to add as their own rows, as {row label: variable name}.
# Use for controls - e.g. a depth-matched Random Forest, so the table shows
# how much of the gap is the algorithm and how much is tree size:
#   BENCH_EXTRA = {"Random Forest (depth 6) + SHAP": "rf_depth6"}

# ============================================================

import os, gc, time, threading, warnings, tempfile
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

for _v in ["X_test", "y_test", "le"]:
    assert _v in globals(), f"{_v} missing - re-run the Phase 2 cells first."

import shap
try:
    import joblib
except ImportError:
    joblib = None

# np.array(..., copy=True): np.asarray on a pandas frame can hand back a
# read-only view under copy-on-write, and the in-place nan_to_num then fails.
X_test_np = np.array(X_test, dtype=np.float64, copy=True)
np.nan_to_num(X_test_np, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
N_FEAT = X_test_np.shape[1]

rs    = np.random.RandomState(42)
e_idx = rs.choice(len(X_test_np), min(N_EXPLAIN, len(X_test_np)), replace=False)
Xe_np = X_test_np[e_idx]
Xe_df = X_test.iloc[e_idx] if hasattr(X_test, "iloc") else Xe_np

# ------------------------------------------------------------------
#  peak-RSS sampler  (tracemalloc misses numpy/C allocations, so sample
#  /proc/self/statm instead and take the high-water mark during the call)
# ------------------------------------------------------------------
_PAGE = os.sysconf("SC_PAGE_SIZE") if hasattr(os, "sysconf") else 4096

def _rss_bytes():
    try:
        with open("/proc/self/statm") as fh:
            return int(fh.read().split()[1]) * _PAGE
    except Exception:
        return 0

class PeakRSS:
    """Samples RSS in a background thread; .peak_mb is the delta over baseline."""
    def __init__(self, interval=0.005):
        self.interval, self.peak, self.base, self._stop = interval, 0, 0, None
    def __enter__(self):
        gc.collect()
        self.base = self.peak = _rss_bytes()
        self._stop = threading.Event()
        def run():
            while not self._stop.is_set():
                r = _rss_bytes()
                if r > self.peak:
                    self.peak = r
                self._stop.wait(self.interval)
        self._t = threading.Thread(target=run, daemon=True); self._t.start()
        return self
    def __exit__(self, *a):
        self._stop.set(); self._t.join(timeout=1.0)
    @property
    def peak_mb(self):
        return max(0.0, (self.peak - self.base) / 1024 / 1024)

def timed(fn):
    """returns (wall_s, cpu_s, peak_rss_mb) for one call"""
    with PeakRSS() as p:
        w0, c0 = time.perf_counter(), time.process_time()
        fn()
        w, c = time.perf_counter() - w0, time.process_time() - c0
    return w, c, p.peak_mb

# ------------------------------------------------------------------
#  adapters: one per detector family
# ------------------------------------------------------------------
class Entry:
    def __init__(self, name, predict, explain, obj, save):
        self.name, self.predict, self.explain = name, predict, explain
        self.obj, self.save = obj, save

def _pin(m):
    """Force single-thread CPU where the estimator exposes it.

    On a GPU runtime an XGBoost model may predict and explain on the GPU while
    the Random Forest runs on CPU. That would make the latency column compare
    hardware, not algorithms, so the device is pinned to CPU explicitly."""
    notes = []
    for attr, val in (("n_jobs", 1), ("nthread", 1), ("n_threads", 1)):
        if hasattr(m, attr):
            try:
                setattr(m, attr, val)
            except Exception:
                notes.append(f"could not pin {attr}")
    if hasattr(m, "device"):                      # XGBoost >= 2.0 sklearn API
        try:
            m.device = "cpu"
        except Exception:
            notes.append("could not set device=cpu on the estimator")
    if hasattr(m, "get_booster"):
        try:
            b = m.get_booster()
            b.set_param({"nthread": 1})
            try:
                b.set_param({"device": "cpu"})    # ignored on older XGBoost
            except Exception:
                pass
            print(f"      xgboost pinned to cpu, nthread=1")
        except Exception:
            notes.append("could not pin booster nthread/device")
    return notes

def _save_sklearn(m, path):
    if joblib is None:
        return float("nan")
    joblib.dump(m, path); return os.path.getsize(path) / 1024

entries, pin_notes, train_sizes = [], [], {}

def _infer_train_n(m):
    """Recover roughly how many rows a tree model was fitted on.

    Random Forest: bootstrap draws n rows WITH replacement, so the root node
    holds about 63.2% distinct rows -> divide by 0.632.
    XGBoost: root 'Cover' is the summed hessian, which scales with n. It is
    only comparable between XGBoost models with the same objective, so it is
    reported raw and flagged as relative."""
    try:
        if hasattr(m, "estimators_") and hasattr(m.estimators_[0], "tree_"):
            root = m.estimators_[0].tree_.n_node_samples[0]
            return ("rows", int(round(root / 0.632)))
        if hasattr(m, "get_booster"):
            df = m.get_booster().trees_to_dataframe()
            return ("cover", float(df[df.Node == 0].Cover.mean()))
    except Exception:
        pass
    return (None, None)

# ---- Random Forest / XGBoost : TreeExplainer ----
def add_tree(name, m):
    pin_notes.extend(f"{name}: {n}" for n in _pin(m))
    kind, val = _infer_train_n(m)
    if kind == "rows":
        train_sizes[name] = val
        print(f"      trained on ~{val:,} rows")
    elif kind == "cover":
        train_sizes[name] = None
        print(f"      root cover {val:,.0f} (relative - compare only with "
              f"other XGBoost models)")
    te = shap.TreeExplainer(m)
    entries.append(Entry(
        name,
        lambda X, m=m: m.predict(X),
        lambda X, te=te: te.shap_values(X, check_additivity=False),
        m, _save_sklearn))

# Detectors are found BY TYPE, not by variable name. Notebooks rename things
# (xgbm / xgb / xgb_model / clf ...) and a name-only lookup silently drops a
# detector, leaving a table that looks complete but is missing a row.
# Candidates must also match X_test's feature count, so a stale model trained
# on different preprocessing cannot sneak in.
def _find(type_substrings, needs_attr=None, exclude_ids=()):
    """all globals whose type name matches, with the right feature count"""
    out = []
    for k, v in list(globals().items()):
        if k.startswith("_") or id(v) in exclude_ids:
            continue
        tn = type(v).__name__
        if not any(s.lower() in tn.lower() for s in type_substrings):
            continue
        if needs_attr and not hasattr(v, needs_attr):
            continue
        nf = getattr(v, "n_features_in_", None)
        if nf is not None and nf != N_FEAT:
            print(f"  skipped {k} ({tn}): trained on {nf} features, "
                  f"X_test has {N_FEAT}")
            continue
        out.append((k, v))
    return out

_seen, _ambiguous = set(), []

def _pick(label, cands, override, hint):
    """one candidate -> use it; several -> REFUSE and make the user choose.

    Two models of the same type are usually the same algorithm trained on
    different numbers of rows (the full-data cell vs the 60k glass-box cell).
    Their latencies are not comparable, and picking the wrong one silently
    produces a table that looks fine and is wrong. So: stop."""
    if override:
        if override not in globals():
            raise SystemExit(f"{label}: BENCH_* names `{override}` but it is "
                             f"not in memory.")
        v = globals()[override]
        print(f"  {label}: using `{override}` (pinned)")
        return v
    if not cands:
        print(f"  NOT FOUND: {label} - {hint}")
        return None
    if len(cands) == 1:
        k, v = cands[0]
        print(f"  {label}: found as `{k}`")
        return v
    names = ", ".join(f"`{k}`" for k, _ in cands)
    print(f"  AMBIGUOUS: {len(cands)} {label} objects in memory - {names}")
    _ambiguous.append((label, [k for k, _ in cands]))
    return None

_rf_v = _pick("Random Forest", _find(["RandomForest"], "feature_importances_"),
              BENCH_RF, "run the cell that trains it in this session.")
if _rf_v is not None:
    add_tree("Random Forest + SHAP", _rf_v); _seen.add(id(_rf_v))

_xg_v = _pick("XGBoost", _find(["XGB"], "feature_importances_", exclude_ids=_seen),
              BENCH_XGB, "the headline detector is missing - run its cell first.")
if _xg_v is not None:
    add_tree("XGBoost + SHAP", _xg_v); _seen.add(id(_xg_v))

for _label, _var in (BENCH_EXTRA or {}).items():
    if _var not in globals():
        raise SystemExit(f"BENCH_EXTRA names `{_var}` but it is not in memory.")
    _m = globals()[_var]
    _nf = getattr(_m, "n_features_in_", None)
    if _nf is not None and _nf != N_FEAT:
        raise SystemExit(f"`{_var}` was trained on {_nf} features, X_test has "
                         f"{N_FEAT}. Not comparable.")
    print(f"  {_label}: using `{_var}` (extra)")
    add_tree(_label, _m); _seen.add(id(_m))

if _ambiguous:
    msg = ["", "=" * 76,
           "STOPPING: more than one model of the same type is in memory.", ""]
    for label, names in _ambiguous:
        msg.append(f"  {label}: {', '.join(names)}")
    msg += ["",
            "These are almost certainly trained on DIFFERENT numbers of rows",
            "(the full-data swap cell vs the 60k glass-box cell). Latencies",
            "across different training sizes are not comparable, so this cell",
            "will not choose for you.",
            "",
            "Set the names at the top of this cell to the ones from the SAME",
            "cell - normally the glass-box cell, since all three detectors",
            "there are trained on identical rows - then re-run:",
            "",
            "    BENCH_RF  = \"<name>\"",
            "    BENCH_XGB = \"<name>\"",
            "=" * 76]
    raise SystemExit("\n".join(msg))

# ---- EBM : eval_terms (the attributions ARE the addends) ----
_ebm_src = None
if BENCH_EBM:
    if BENCH_EBM not in globals():
        raise SystemExit(f"BENCH_EBM names `{BENCH_EBM}` but it is not in memory.")
    _ebm_src = globals()[BENCH_EBM]
    print(f"  EBM: using `{BENCH_EBM}` (pinned)")
for _nm in () if _ebm_src else ("ebms", "models"):
    if _nm in globals() and isinstance(globals()[_nm], dict) and globals()[_nm]:
        if all(hasattr(m, "eval_terms") for m in globals()[_nm].values()):
            _ebm_src = globals()[_nm]
            print(f"  found EBM dict as `{_nm}` "
                  f"(K = {sorted(_ebm_src)})")
            break
_ebm_named = None
if _ebm_src is None:
    _eb = _find(["Explainable", "EBM"], "eval_terms", exclude_ids=_seen)
    if _eb:
        _ebm_named = _eb
        print(f"  found {len(_eb)} EBM(s) by type: {[k for k, _ in _eb]}")
    else:
        print("  NOT FOUND: EBM - run the glassbox cell in this session.")

def _ebm_K(m, fallback=None):
    """read the interaction count off the model, not off the dict key"""
    v = getattr(m, "interactions", None)
    if isinstance(v, int):
        return v
    if isinstance(v, (list, tuple)):
        return len(v)
    return fallback

if _ebm_src:
    for key in sorted(_ebm_src):
        m = _ebm_src[key]
        K = _ebm_K(m, key)
        entries.append(Entry(f"EBM glass-box (K={K})",
                             lambda X, m=m: m.predict(X),
                             lambda X, m=m: m.eval_terms(X),
                             m, _save_sklearn))
elif _ebm_named:
    for nm, m in _ebm_named:
        K = _ebm_K(m)
        label = f"EBM glass-box (K={K})" if K is not None else f"EBM glass-box [{nm}]"
        entries.append(Entry(label,
                             lambda X, m=m: m.predict(X),
                             lambda X, m=m: m.eval_terms(X),
                             m, _save_sklearn))
elif "ebm" in globals():
    m = globals()["ebm"]
    entries.append(Entry("EBM glass-box",
                         lambda X, m=m: m.predict(X),
                         lambda X, m=m: m.eval_terms(X),
                         m, _save_sklearn))

# ---- DNN : DeepExplainer (needs the fitted scaler from the DNN cell) ----
# Gated on the DNN_READY sentinel, not on `model`/`scaler` merely existing.
# A notebook can easily hold an unrelated variable called `model`, and pairing
# it with someone else's `scaler` would produce a plausible-looking row that
# describes nothing. Absent or mismatched sentinel -> skip the DNN, loudly.
_dnn_ok = False
if "DNN_READY" in globals() and isinstance(globals()["DNN_READY"], dict):
    _s = globals()["DNN_READY"]
    if _s.get("dataset") != DATASET:
        print(f"  DNN SKIPPED: dnn_shap.py ran on {_s.get('dataset')!r} but this "
              f"cell is set to {DATASET!r}. Re-run dnn_shap.py for this dataset.")
    elif _s.get("n_feat") != N_FEAT:
        print(f"  DNN SKIPPED: DNN was built for {_s.get('n_feat')} features, "
              f"X_test has {N_FEAT}. Different preprocessing - do not compare.")
    elif _s.get("explainer") == "unavailable":
        print("  DNN SKIPPED: dnn_shap.py could not build an explainer.")
    elif "model" not in globals() or "scaler" not in globals():
        print("  DNN SKIPPED: `model`/`scaler` are gone from memory.")
    else:
        _dnn_ok = True
elif "model" in globals():
    print("  DNN SKIPPED: found a `model` variable but no DNN_READY sentinel. "
          "Run dnn_shap.py in this session first - an unrelated `model` will "
          "not be benchmarked.")

if _dnn_ok:
    try:
        import torch
        torch.set_num_threads(1)
        _dnn, _sc = globals()["model"], globals()["scaler"]
        _dnn.eval()
        _Xe_s  = _sc.transform(Xe_np).astype(np.float32)
        _Xe_t  = torch.from_numpy(_Xe_s)

        # DeepSHAP cost scales with the background set, so this MUST match the
        # background dnn_shap.py used or the two cells report different numbers
        # for the same model. Inherit DEEP_BG and draw from TRAIN, as it did.
        _nbg = int(globals().get("DEEP_BG", 100))
        if "X_train" in globals():
            _src = np.array(globals()["X_train"], dtype=np.float64, copy=True)
            np.nan_to_num(_src, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        else:
            _src = X_test_np
            print("  NOTE: X_train absent - DeepSHAP background drawn from test "
                  "set, so this number will not match the dnn_shap.py one")
        _bg_rows = _src[np.random.RandomState(42).choice(len(_src),
                                                         min(_nbg, len(_src)),
                                                         replace=False)]
        _bg_np = _sc.transform(_bg_rows).astype(np.float32)
        print(f"  DNN DeepSHAP background: {len(_bg_np)} rows "
              f"(DEEP_BG={_nbg}) - must match dnn_shap.py")
        _de    = shap.DeepExplainer(_dnn, torch.from_numpy(_bg_np))
        _de.shap_values(_Xe_t[:4], check_additivity=False)       # warm up

        def _dnn_pred(X, _dnn=_dnn, _sc=_sc):
            A = _sc.transform(np.asarray(X, dtype=np.float64)).astype(np.float32)
            with torch.no_grad():
                return _dnn(torch.from_numpy(A)).argmax(1).numpy()

        def _dnn_expl(X, _de=_de, _sc=_sc):
            A = _sc.transform(np.asarray(X, dtype=np.float64)).astype(np.float32)
            return _de.shap_values(torch.from_numpy(A), check_additivity=False)

        def _save_torch(m, path, torch=torch):
            torch.save(m.state_dict(), path); return os.path.getsize(path) / 1024

        entries.append(Entry("DNN + DeepSHAP", _dnn_pred, _dnn_expl, _dnn, _save_torch))
    except Exception as e:
        print(f"(DNN skipped: {type(e).__name__}: {e})")

if not entries:
    raise SystemExit("No detectors in memory. Run the comparison cell "
                     "(and optionally dnn_shap.py) in this session first.")

# Cross-check training sizes: a mismatch invalidates the latency comparison,
# because SHAP cost scales with tree size and tree size scales with rows.
_known = {k: v for k, v in train_sizes.items() if v}
if "DNN_READY" in globals() and isinstance(globals()["DNN_READY"], dict):
    _n = globals()["DNN_READY"].get("n_train")
    if _n:
        _known["DNN + DeepSHAP"] = _n
if len(_known) > 1:
    _lo, _hi = min(_known.values()), max(_known.values())
    print("\n  training sizes:",
          ", ".join(f"{k.split(' +')[0]} ~{v:,}" for k, v in _known.items()))
    if _hi > _lo * 1.5:
        print("\n" + "!" * 76)
        print("  WARNING: these models were trained on very different amounts "
              "of data.")
        print("  SHAP cost scales with tree size, which scales with training "
              "rows, so")
        print("  the latency column below compares training-set sizes, not "
              "algorithms.")
        print("  Retrain them on the same rows before quoting this table.")
        print("!" * 76)

print(f"{DATASET}: benchmarking {len(entries)} detector(s) on {len(Xe_np):,} flows, "
      f"{ROUNDS} interleaved rounds\n")
for n in pin_notes:
    print(f"  WARNING  {n}")

# ------------------------------------------------------------------
#  warm up everything once, outside the timing loop
# ------------------------------------------------------------------
for e in entries:
    try:
        e.predict(Xe_df[:8] if hasattr(Xe_df, "iloc") else Xe_np[:8])
        e.explain(Xe_df[:4] if hasattr(Xe_df, "iloc") else Xe_np[:4])
    except Exception as ex:
        print(f"  warm-up failed for {e.name}: {type(ex).__name__}: {ex}")

# ------------------------------------------------------------------
#  interleaved measurement
# ------------------------------------------------------------------
acc = {e.name: {"inf_w": [], "inf_c": [], "exp_w": [], "exp_c": [],
                "exp_mb": [], "b1_w": []} for e in entries}

for r in range(ROUNDS):
    for e in entries:
        X = Xe_df if hasattr(Xe_df, "iloc") else Xe_np
        try:
            w, c, _ = timed(lambda e=e, X=X: e.predict(X))
            acc[e.name]["inf_w"].append(1000 * w / len(Xe_np))
            acc[e.name]["inf_c"].append(1000 * c / len(Xe_np))

            w, c, mb = timed(lambda e=e, X=X: e.explain(X))
            acc[e.name]["exp_w"].append(1000 * w / len(Xe_np))
            acc[e.name]["exp_c"].append(1000 * c / len(Xe_np))
            acc[e.name]["exp_mb"].append(mb)

            n1 = min(N_BATCH1, len(Xe_np))
            def _b1(e=e, X=X, n1=n1):
                for i in range(n1):
                    e.predict(X.iloc[i:i + 1] if hasattr(X, "iloc") else X[i:i + 1])
            w, c, _ = timed(_b1)
            acc[e.name]["b1_w"].append(1000 * w / n1)
        except Exception as ex:
            print(f"  round {r+1} {e.name}: {type(ex).__name__}: {ex}")
    print(f"  round {r + 1}/{ROUNDS} done")

# ------------------------------------------------------------------
#  model size on disk
# ------------------------------------------------------------------
sizes = {}
with tempfile.TemporaryDirectory() as td:
    for e in entries:
        try:
            sizes[e.name] = e.save(e.obj, os.path.join(td, "m.bin"))
        except Exception:
            sizes[e.name] = float("nan")

# ------------------------------------------------------------------
#  results
# ------------------------------------------------------------------
def med(v):
    return float(np.median(v)) if v else float("nan")

rows = []
for e in entries:
    a = acc[e.name]
    iw, ew = med(a["inf_w"]), med(a["exp_w"])
    rows.append({
        "Model":            e.name,
        "Inference (ms)":   round(iw, 4),
        "Inf CPU (ms)":     round(med(a["inf_c"]), 4),
        "Explain (ms)":     round(ew, 4),
        "Exp CPU (ms)":     round(med(a["exp_c"]), 4),
        "End-to-end (ms)":  round(iw + ew, 4),
        "Flows/sec":        round(1000 / (iw + ew), 1) if (iw + ew) > 0 else float("nan"),
        "Batch-1 (ms)":     round(med(a["b1_w"]), 4),
        "Peak RSS (MB)":    round(med(a["exp_mb"]), 1),
        "Size (KB)":        round(sizes.get(e.name, float("nan")), 1),
    })

t = pd.DataFrame(rows)
print("\n" + "=" * 128)
print(f"{DATASET} - RESOURCE BENCHMARK (medians over {ROUNDS} interleaved rounds, "
      f"1 thread)")
print("=" * 128)
print(t.to_string(index=False))
print("=" * 128)

print("\nSpread across rounds (min - max wall ms/flow for the explanation step):")
for e in entries:
    v = acc[e.name]["exp_w"]
    if v:
        print(f"  {e.name:26s} {min(v):8.4f} - {max(v):8.4f}  "
              f"(cpu/wall ratio {med(acc[e.name]['exp_c']) / max(med(v), 1e-9):.2f})")

print("""
READING THIS TABLE
  CPU vs wall: a ratio near 1.0 means the model is genuinely single-threaded.
  Above 1.0 means it is still using several cores and the wall-clock number
  flatters it - pin the threads and re-run before quoting.

  Batch-1 is the number a live IDS actually experiences. Batched throughput
  is the number papers usually report; the gap between them is worth a line.

  Peak RSS is the high-water mark over baseline DURING the explanation call,
  sampled every 5 ms. It is a floor, not an exact figure - a short spike
  between samples is missed. Report it as approximate.

  The DNN row will read slightly HIGHER here than in dnn_shap.py (~10%): this
  cell times the scaler transform as part of the call, because feature scaling
  is part of the deployed pipeline, while dnn_shap.py times the explainer on
  already-scaled input. Quote THIS table for the end-to-end comparison and say
  so once in the methods section. If the gap is much larger than ~10%, the
  background sizes have diverged - check the DEEP_BG line printed above.
""")

TAG  = DATASET.lower().replace(" ", "")
DIRS = ["/content/artifacts", "/content/drive/MyDrive/CN_PROJECT/artifacts"]
for d in DIRS:
    try:
        os.makedirs(d, exist_ok=True)
        t.to_csv(f"{d}/resource_benchmark_{TAG}.csv", index=False)
    except Exception as ex:
        print(f"(skipped {d}: {ex})")

log = f"[RESOURCE/{DATASET}] " + " | ".join(
    f"{r['Model']}: inf {r['Inference (ms)']:.3f} exp {r['Explain (ms)']:.3f} "
    f"cpu {r['Exp CPU (ms)']:.3f} rss {r['Peak RSS (MB)']:.1f}MB "
    f"size {r['Size (KB)']:.0f}KB" for r in rows)
for d in DIRS:
    try:
        with open(f"{d}/results_log.txt", "a") as fh:
            fh.write(log + "\n")
    except Exception:
        pass
print("\nPASTE INTO RESULTS LOG:\n ", log)

  Random Forest: using `rf_full` (pinned)
      trained on ~149,840 rows
  XGBoost: using `xgb` (pinned)
      xgboost pinned to cpu, nthread=1
      root cover 2,875 (relative - compare only with other XGBoost models)
  Random Forest (depth 6) + SHAP: using `rf_depth6` (extra)
      trained on ~149,840 rows
  NOT FOUND: EBM - run the glassbox cell in this session.
  DNN DeepSHAP background: 100 rows (DEEP_BG=100) - must match dnn_shap.py

  training sizes: Random Forest ~149,840, Random Forest (depth 6) ~149,840, DNN ~150,000
InSDN: benchmarking 4 detector(s) on 500 flows, 3 interleaved rounds

  round 1/3 done
  round 2/3 done
  round 3/3 done

InSDN - RESOURCE BENCHMARK (medians over 3 interleaved rounds, 1 thread)
                         Model  Inference (ms)  Inf CPU (ms)  Explain (ms)  Exp CPU (ms)  End-to-end (ms)  Flows/sec  Batch-1 (ms)  Peak RSS (MB)  Size (KB)
          Random Forest + SHAP          0.0290        0.0300       13.0423       13.1064          13.0713       76.

In [ ]:
# ============================================================
# GLASS-BOX EVALUATION - ONE CELL, ANY DATASET
#   RF + SHAP   vs   XGBoost + SHAP   vs   EBM glass-box
#   + interaction sweep (the interpretability/accuracy knob)
#   + interleaved latency benchmark (all models, one session)
#
# Paste as ONE cell after the Phase 2 cells of ANY dataset notebook
# (InSDN now, CICIoT2023 later). Set DATASET below.
#
# Needs in memory: X_train, X_test, y_train, y_test, le
#
# Everything is trained on the SAME rows and timed in the SAME session,
# so there is no cross-session confound to explain away.
#
# Runtime: ~12-20 min, dominated by EBM training. Trim K_VALUES if slow.
# ============================================================

DATASET  = "InSDN"          # <-- change for other datasets
FIT_N    = 60000            # rows all models train on (None = use everything)
K_VALUES = [0, 10, 30]      # EBM pairwise-interaction counts to sweep
ROUNDS   = 200              # interleaved latency rounds
# ============================================================

import importlib, subprocess, sys
for _pkg, _pip in (("xgboost", "xgboost"), ("shap", "shap"),
                   ("interpret", "interpret-core")):
    if importlib.util.find_spec(_pkg) is None:
        print(f"installing {_pip} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pip],
                       check=True)

import numpy as np, pandas as pd, time, os, joblib, shap
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from interpret.glassbox import ExplainableBoostingClassifier

for _v in ["X_train", "X_test", "y_train", "y_test", "le"]:
    assert _v in globals(), f"{_v} missing - re-run the Phase 2 cells first."

os.makedirs("/content/artifacts", exist_ok=True)
DRIVE_DIR = "/content/drive/MyDrive/CN_PROJECT/artifacts"
TAG = DATASET.lower().replace(" ", "_")

N_CLASSES  = len(le.classes_)
N_FEATURES = X_train.shape[1]
FEATS      = list(X_train.columns)
y_test_arr = np.asarray(y_test)

# ---- the same rows for every model ----
if FIT_N is not None and FIT_N < len(X_train):
    idx  = np.random.default_rng(42).choice(len(X_train), FIT_N, replace=False)
    Xtr  = X_train.iloc[idx].reset_index(drop=True)
    ytr  = np.asarray(y_train)[idx]
    print(f"!! all models trained on the SAME {FIT_N:,} rows (of "
          f"{len(X_train):,}). Not comparable to full-data numbers.")
else:
    Xtr, ytr = X_train.reset_index(drop=True), np.asarray(y_train)
    print(f"all models trained on the full {len(Xtr):,} training rows.")

sw = compute_sample_weight(class_weight="balanced", y=ytr)
print(f"\n{DATASET}: {len(Xtr):,} train / {len(X_test):,} test | "
      f"{N_FEATURES} features | {N_CLASSES} classes")

# ---- the two rarest classes carry the macro-F1 story; find them generically ----
support = pd.Series(y_test_arr).value_counts().sort_values()
rare_ids = list(support.index[:2])
rare_names = [le.classes_[i] for i in rare_ids]
print(f"two rarest test classes: "
      + ", ".join(f"{n} (n={support[i]:,})" for n, i in zip(rare_names, rare_ids)))
print()


def evaluate(model, Xte=None):
    yp = model.predict(X_test if Xte is None else Xte)
    rep = classification_report(y_test_arr, yp, labels=np.arange(N_CLASSES),
                                target_names=list(le.classes_),
                                output_dict=True, zero_division=0)
    return (accuracy_score(y_test_arr, yp),
            f1_score(y_test_arr, yp, average="macro", zero_division=0),
            rep)


# ============================================================
# TRAIN
# ============================================================
print("[1/3] Random Forest ...")
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                            n_jobs=-1, random_state=42)
t0 = time.time(); rf.fit(Xtr, ytr); rf_fit = time.time() - t0
rf_acc, rf_f1, rf_rep = evaluate(rf)
print(f"      acc {rf_acc:.4f} | macro-F1 {rf_f1:.4f} | fit {rf_fit:.0f}s")

print("[2/3] XGBoost ...")
xgbm = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.3,
                     tree_method="hist", n_jobs=-1,
                     objective="multi:softprob", num_class=N_CLASSES,
                     random_state=42, verbosity=0)
t0 = time.time(); xgbm.fit(Xtr, ytr, sample_weight=sw); xgb_fit = time.time() - t0
xgb_acc, xgb_f1, xgb_rep = evaluate(xgbm)
print(f"      acc {xgb_acc:.4f} | macro-F1 {xgb_f1:.4f} | fit {xgb_fit:.0f}s")

print(f"[3/3] EBM glass-box, K in {K_VALUES} ...")
ebms, sweep = {}, []
for K in K_VALUES:
    m = ExplainableBoostingClassifier(interactions=K, max_rounds=200,
                                      outer_bags=4, n_jobs=-1, random_state=42)
    t0 = time.time(); m.fit(Xtr, ytr, sample_weight=sw); fit_s = time.time() - t0
    a, f, rep = evaluate(m)
    n_terms = len(m.term_features_)
    n_pairs = sum(1 for tf in m.term_features_ if len(tf) > 1)
    if K == 0:
        assert n_terms == N_FEATURES, \
            f"K=0 built {n_terms} terms for {N_FEATURES} features"
    row = {"K": K, "terms": n_terms, "pairs": n_pairs,
           "Accuracy": round(a, 4), "Macro-F1": round(f, 4), "fit s": round(fit_s)}
    for n in rare_names:
        row[f"{n} F1"]   = round(rep[n]["f1-score"], 3)
        row[f"{n} prec"] = round(rep[n]["precision"], 3)
    sweep.append(row); ebms[K] = m
    print(f"      K={K:3d}  macro-F1 {f:.4f} | "
          + " | ".join(f"{n} F1 {rep[n]['f1-score']:.3f}" for n in rare_names)
          + f" | {n_pairs} pairs | fit {fit_s:.0f}s")

sweep_df = pd.DataFrame(sweep)
best_K   = int(sweep_df.loc[sweep_df["Macro-F1"].idxmax(), "K"])
# knee = smallest K within 0.2pp of the best
near = sweep_df[sweep_df["Macro-F1"] >= sweep_df["Macro-F1"].max() - 0.002]
knee_K = int(near.loc[near["K"].idxmin(), "K"])


# ============================================================
# INTERLEAVED LATENCY - every model, same session, same flows
# ============================================================
print(f"\nlatency: {ROUNDS} interleaved rounds ...")
cand = [{"name": "Random Forest + SHAP", "m": rf,   "kind": "tree"},
        {"name": "XGBoost + SHAP",       "m": xgbm, "kind": "tree"}]
for K in K_VALUES:
    cand.append({"name": f"EBM glass-box (K={K})", "m": ebms[K], "kind": "ebm"})

for c in cand:
    if c["kind"] == "tree":
        c["ex"] = shap.TreeExplainer(c["m"])
        c["explain"] = lambda r, c=c: c["ex"].shap_values(r)
    else:
        c["explain"] = lambda r, c=c: c["m"].eval_terms(r)
    c["predict"] = lambda r, c=c: c["m"].predict(r)

pool = [X_test.iloc[[i]] for i in range(min(200, len(X_test)))]
for i in range(10):                                  # warm-up, discarded
    r = pool[i % len(pool)]
    for c in cand: c["predict"](r); c["explain"](r)

inf = {c["name"]: [] for c in cand}
xai = {c["name"]: [] for c in cand}
for rnd in range(ROUNDS):
    r = pool[rnd % len(pool)]
    for c in cand:                                   # round-robin
        t = time.perf_counter(); c["predict"](r)
        inf[c["name"]].append((time.perf_counter() - t) * 1000)
        t = time.perf_counter(); c["explain"](r)
        xai[c["name"]].append((time.perf_counter() - t) * 1000)
    if (rnd + 1) % 100 == 0: print(f"  {rnd+1}/{ROUNDS}")

def q(a):
    a = np.asarray(a)
    return float(np.median(a)), float(np.percentile(a,25)), float(np.percentile(a,75))

lat = []
for c in cand:
    im, _, _    = q(inf[c["name"]])
    xm, x25, x75 = q(xai[c["name"]])
    e2e = np.asarray(inf[c["name"]]) + np.asarray(xai[c["name"]])
    em, e25, e75 = q(e2e)
    lat.append({"Model": c["name"], "Inference ms": round(im, 3),
                "Explain ms": round(xm, 3), "Explain IQR": f"{x25:.2f}-{x75:.2f}",
                "End-to-end ms": round(em, 3), "E2E IQR": f"{e25:.2f}-{e75:.2f}",
                "Flows/sec": round(1000/em, 1), "_em": em, "_e25": e25, "_e75": e75})
lat_df = pd.DataFrame(lat)


# ============================================================
# REPORT
# ============================================================
print("\n" + "=" * 112)
print(f"{DATASET} - DETECTION, {len(Xtr):,} training rows, "
      f"{len(X_test):,} test flows")
print("=" * 112)
det = pd.DataFrame([
    {"Model": "Random Forest + SHAP", "Explanation": "post-hoc (approx.)",
     "Accuracy": round(rf_acc, 4),  "Macro-F1": round(rf_f1, 4)},
    {"Model": "XGBoost + SHAP",      "Explanation": "post-hoc (approx.)",
     "Accuracy": round(xgb_acc, 4), "Macro-F1": round(xgb_f1, 4)},
    {"Model": f"EBM glass-box (K={knee_K})", "Explanation": "built in (exact)",
     "Accuracy": sweep_df.loc[sweep_df.K == knee_K, "Accuracy"].iloc[0],
     "Macro-F1": sweep_df.loc[sweep_df.K == knee_K, "Macro-F1"].iloc[0]},
])
print(det.to_string(index=False))

print("\n" + "=" * 112)
print(f"{DATASET} - EBM INTERACTION SWEEP")
print("=" * 112)
print(sweep_df.to_string(index=False))
print(f"\npeak macro-F1 at K={best_K}; recommended knee K={knee_K}")

print("\n" + "=" * 112)
print(f"{DATASET} - PER-FLOW LATENCY, {ROUNDS} interleaved rounds, median [IQR]")
print("=" * 112)
print(lat_df.drop(columns=["_em","_e25","_e75"]).to_string(index=False))
print("=" * 112)

ebm_row = lat_df[lat_df.Model == f"EBM glass-box (K={knee_K})"].iloc[0]
knee_f1 = sweep_df.loc[sweep_df.K == knee_K, "Macro-F1"].iloc[0]
print(f"\nEBM (K={knee_K}) vs the post-hoc baselines:")
for _, b in lat_df[~lat_df.Model.str.startswith("EBM")].iterrows():
    base_f1 = rf_f1 if "Forest" in b["Model"] else xgb_f1
    print(f"  vs {b['Model']:22s} "
          f"{b['_em']/ebm_row['_em']:5.1f}x cheaper end-to-end "
          f"(conservative {b['_e25']/ebm_row['_e75']:.1f}x), "
          f"macro-F1 {(knee_f1-base_f1)*100:+.2f} pp")

print("\nREPLICATION CHECK vs CICIDS2017 (K=0 -> K=30 was +3.67 pp there):")
if 0 in ebms and 30 in [r['K'] for r in sweep]:
    d = (sweep_df.loc[sweep_df.K == 30, "Macro-F1"].iloc[0]
         - sweep_df.loc[sweep_df.K == 0, "Macro-F1"].iloc[0]) * 100
    print(f"  here K=0 -> K=30 is {d:+.2f} pp.")
    print("  Same direction and similar size = the additive-constraint finding")
    print("  replicates across datasets, which is worth a sentence in the paper.")
    print("  Different = report both and do NOT generalise from one dataset.")

print("\nCAVEAT TO STATE IN THE PAPER: these are single-flow latencies including")
print("Python/framework overhead, which is the real per-alert cost but is not")
print("purely algorithmic. Batched throughput would narrow the gaps.")


# ============================================================
# LaTeX + SAVE
# ============================================================
print(f"\nLaTeX ({DATASET}):\n")
print("\\begin{table}[t]\n\\centering")
print(f"\\caption{{Glass-box vs post-hoc explanation on {DATASET}. "
      f"All detectors trained on {len(Xtr):,} flows.}}")
print(f"\\label{{tab:glassbox-{TAG}}}")
print("\\begin{tabular}{llcccc}\n\\hline")
print("Model & Explanation & Acc. & Macro-F1 & Explain (ms) & End-to-end (ms) \\\\")
print("\\hline")
for _, d in det.iterrows():
    L = lat_df[lat_df.Model.str.startswith(d["Model"].split(" (")[0])]
    L = L.iloc[-1] if len(L) else None
    if L is None: continue
    print(f"{d['Model']} & {d['Explanation']} & {d['Accuracy']:.4f} & "
          f"{d['Macro-F1']:.4f} & {L['Explain ms']:.2f} & "
          f"{L['End-to-end ms']:.2f} \\\\")
print("\\hline\n\\end{tabular}\n\\end{table}")

for d in ["/content/artifacts", DRIVE_DIR]:
    try:
        os.makedirs(d, exist_ok=True)
        det.to_csv(f"{d}/glassbox_detection_{TAG}.csv", index=False)
        sweep_df.to_csv(f"{d}/glassbox_sweep_{TAG}.csv", index=False)
        lat_df.drop(columns=["_em","_e25","_e75"]).to_csv(
            f"{d}/glassbox_latency_{TAG}.csv", index=False)
        joblib.dump(ebms[knee_K], f"{d}/ebm_{TAG}_K{knee_K}.pkl")
    except Exception as e:
        print(f"(skipped {d}: {e})")

log = (f"[GLASSBOX-{DATASET}] n={len(Xtr)} | RF F1 {rf_f1:.4f} | "
       f"XGB F1 {xgb_f1:.4f} | EBM K={knee_K} F1 {knee_f1:.4f} | "
       f"EBM e2e {ebm_row['End-to-end ms']:.2f}ms IQR {ebm_row['E2E IQR']} | "
       f"sweep " + " ".join(f"K{r['K']}:{r['Macro-F1']:.4f}" for r in sweep))
for d in ["/content/artifacts", DRIVE_DIR]:
    try:
        with open(f"{d}/results_log.txt", "a") as fh: fh.write(log + "\n")
    except Exception: pass
print("\nPASTE INTO RESULTS LOG:\n ", log)

installing interpret-core ...
!! all models trained on the SAME 60,000 rows (of 150,000). Not comparable to full-data numbers.

InSDN: 60,000 train / 50,000 test | 79 features | 5 classes
two rarest test classes: Brute Force (n=72), DoS (n=279)

[1/3] Random Forest ...
      acc 0.9997 | macro-F1 0.9872 | fit 4s
[2/3] XGBoost ...
      acc 0.9999 | macro-F1 0.9955 | fit 6s
[3/3] EBM glass-box, K in [0, 10, 30] ...


KeyboardInterrupt: 

In [ ]:
# ============================================================
# FULL CLASSIFICATION METRICS for the algorithm comparison
#   Accuracy · Macro-Precision · Macro-Recall · Macro-F1
#   (+ weighted versions, which is what "accuracy-like" really is)
#
# Paste as ONE cell in the SAME session that ran the comparison, so the
# trained models are still in memory. Nothing is retrained.
# Needs: X_test, y_test, le  and whichever of rf / xgbm / xgb / ebms / ebm exist.
#
# Runtime: seconds.
# ============================================================

DATASET = "CICIDS2017"      # label only, for the printout and filename
# ============================================================

import numpy as np, pandas as pd, os
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)

for _v in ["X_test", "y_test", "le"]:
    assert _v in globals(), f"{_v} missing - re-run the Phase 2 cells first."

y_true = np.asarray(y_test)
N_FEATURES = X_test.shape[1]

# ---- collect whatever detectors are still in memory ----
found = []
if "rf" in globals():
    found.append(("Random Forest + SHAP", globals()["rf"]))
for v in ("xgbm", "xgb"):
    if v in globals() and hasattr(globals()[v], "feature_importances_"):
        found.append(("XGBoost + SHAP", globals()[v])); break
if "ebms" in globals() and isinstance(globals()["ebms"], dict):
    for K in sorted(globals()["ebms"]):
        found.append((f"EBM glass-box (K={K})", globals()["ebms"][K]))
elif "models" in globals() and isinstance(globals()["models"], dict):
    for K in sorted(globals()["models"]):
        found.append((f"EBM glass-box (K={K})", globals()["models"][K]))
elif "ebm" in globals():
    found.append(("EBM glass-box", globals()["ebm"]))

if not found:
    raise SystemExit(
        "No trained detectors in memory. Re-run the comparison cell "
        "(glassbox_full.py / glassbox_model.py) in this session first.")

print(f"{DATASET}: scoring {len(found)} detector(s) on {len(X_test):,} test flows\n")

rows, reports = [], {}
for name, m in found:
    # the EBM trained on a feature subset would need the same columns; all of
    # ours used every feature, but guard anyway
    n_in = getattr(m, "n_features_in_", N_FEATURES)
    Xte = X_test if n_in == N_FEATURES else X_test.iloc[:, :n_in]
    yp = m.predict(Xte)
    rows.append({
        "Model": name,
        "Accuracy":        round(accuracy_score(y_true, yp), 4),
        "Macro-Precision": round(precision_score(y_true, yp, average="macro",
                                                 zero_division=0), 4),
        "Macro-Recall":    round(recall_score(y_true, yp, average="macro",
                                              zero_division=0), 4),
        "Macro-F1":        round(f1_score(y_true, yp, average="macro",
                                          zero_division=0), 4),
        "Wtd-Precision":   round(precision_score(y_true, yp, average="weighted",
                                                 zero_division=0), 4),
        "Wtd-Recall":      round(recall_score(y_true, yp, average="weighted",
                                              zero_division=0), 4),
        "Wtd-F1":          round(f1_score(y_true, yp, average="weighted",
                                          zero_division=0), 4),
    })
    reports[name] = classification_report(
        y_true, yp, labels=np.arange(len(le.classes_)),
        target_names=list(le.classes_), zero_division=0)
    print(f"  {name:28s} acc {rows[-1]['Accuracy']:.4f} | "
          f"P {rows[-1]['Macro-Precision']:.4f} | "
          f"R {rows[-1]['Macro-Recall']:.4f} | "
          f"F1 {rows[-1]['Macro-F1']:.4f}")

t = pd.DataFrame(rows)

print("\n" + "=" * 112)
print(f"{DATASET} - MACRO metrics (every class counts equally)")
print("=" * 112)
print(t[["Model", "Accuracy", "Macro-Precision", "Macro-Recall",
         "Macro-F1"]].to_string(index=False))
print("=" * 112)

print("\n" + "=" * 112)
print(f"{DATASET} - WEIGHTED metrics (classes counted by how common they are)")
print("=" * 112)
print(t[["Model", "Accuracy", "Wtd-Precision", "Wtd-Recall",
         "Wtd-F1"]].to_string(index=False))
print("=" * 112)

print("""
WHICH TO QUOTE
  Macro treats a 193-flow class the same as a 207,781-flow one, so it is the
  honest metric on an imbalanced dataset and the one that separates these
  models. Weighted and accuracy are dominated by the majority class and will
  look near-identical across all three - say so rather than letting someone
  think the models are equivalent.

  If macro-Recall is high while macro-Precision is low, the model is
  over-predicting the rare classes: it catches them but floods you with false
  alarms. That is exactly the EBM's failure mode on the rarest classes.
""")

print("\nPer-class detail (for the appendix - report support counts alongside):")
for name, rep in reports.items():
    print("\n" + "-" * 72)
    print(name)
    print("-" * 72)
    print(rep)

TAG = DATASET.lower().replace(" ", "")
for d in ["/content/artifacts", "/content/drive/MyDrive/CN_PROJECT/artifacts"]:
    try:
        os.makedirs(d, exist_ok=True)
        t.to_csv(f"{d}/full_metrics_{TAG}.csv", index=False)
    except Exception as e:
        print(f"(skipped {d}: {e})")

log = f"[FULL-METRICS/{DATASET}] " + " | ".join(
    f"{r['Model']}: acc {r['Accuracy']:.4f} P {r['Macro-Precision']:.4f} "
    f"R {r['Macro-Recall']:.4f} F1 {r['Macro-F1']:.4f}" for r in rows)
for d in ["/content/artifacts", "/content/drive/MyDrive/CN_PROJECT/artifacts"]:
    try:
        with open(f"{d}/results_log.txt", "a") as fh: fh.write(log + "\n")
    except Exception: pass
print("\nPASTE INTO RESULTS LOG:\n ", log)

CICIDS2017: scoring 2 detector(s) on 50,000 test flows

  Random Forest + SHAP         acc 0.9998 | P 0.9963 | R 0.9861 | F1 0.9910
  EBM glass-box                acc 0.9999 | P 0.9912 | R 0.9999 | F1 0.9955

CICIDS2017 - MACRO metrics (every class counts equally)
               Model  Accuracy  Macro-Precision  Macro-Recall  Macro-F1
Random Forest + SHAP    0.9998           0.9963        0.9861    0.9910
       EBM glass-box    0.9999           0.9912        0.9999    0.9955

CICIDS2017 - WEIGHTED metrics (classes counted by how common they are)
               Model  Accuracy  Wtd-Precision  Wtd-Recall  Wtd-F1
Random Forest + SHAP    0.9998         0.9998      0.9998  0.9998
       EBM glass-box    0.9999         0.9999      0.9999  0.9999

WHICH TO QUOTE
  Macro treats a 193-flow class the same as a 207,781-flow one, so it is the
  honest metric on an imbalanced dataset and the one that separates these
  models. Weighted and accuracy are dominated by the majority class and will
  loo

In [ ]:
import joblib, torch, os
D = "/content/drive/MyDrive/CN_PROJECT/artifacts/models_insdn_full"
os.makedirs(D, exist_ok=True)

for n in ["rf_full", "rf_depth6", "xgb", "scaler"]:
    if n in globals():
        joblib.dump(globals()[n], f"{D}/{n}.joblib")
        print("saved", n)
    else:
        print("missing", n)

if "model" in globals():
    torch.save(model.state_dict(), f"{D}/dnn.pt")
    print("saved dnn state_dict")

print("\nin", D, "->", sorted(os.listdir(D)))

missing rf_full
missing rf_depth6
missing xgb
missing scaler

in /content/drive/MyDrive/CN_PROJECT/artifacts/models_insdn_full -> []
